# Trustworthy Cross-Regional Onion Disease Diagnosis — Full Pipeline (Kaggle)

One notebook, end to end. It reads the existing Kaggle dataset **`onion-disease-multi-source`** (output of Step 0) and produces every result the paper needs:

| Section | What it does | RQ |
|---|---|---|
| 1–3 | Find data, canonicalize Roboflow copies, cache images, exact + perceptual-hash dedup, label conflicts | rules |
| 4 | Harmonized label map + visual sanity grid (incl. *Iris yellow virus* vs *Virosis-D*) | rules |
| 5 | Group-aware, leakage-checked splits: **leave-one-dataset-out (LODO)**, inner **leave-one-source-out** (calibration), **stratified group 5-fold CV** | RQ1, RQ3 |
| 6 | Models: 5 architectures + 3 frozen **foundation models** + 3 **domain-generalization** baselines + proposed **SILA-Net** + ablations | RQ1, RQ2 |
| 7 | Training engine (GPU augmentation, AMP, SWA, foundation-model linear probes, resumable) | — |
| 8 | Run all experiments (priority order, time guard) | RQ1, RQ2 |
| 9 | Generalization results, degradation, statistics (Wilcoxon, McNemar, bootstrap CI) | RQ1, RQ2 |
| 10 | Calibration; conformal prediction with 3 calibration sources (`val`, `worst`, **cross-source**) × 3 gates (disease, remedy-aggregated, **remedy-level CP**); **wrong remedies avoided** | RQ3 |
| 11 | Quantitative XAI: deletion/insertion, saliency concentration, correct vs wrong, lesion-mask metrics, annotation pack | RQ4 |
| 12 | Region × season remedy knowledge base + uncertainty-gated recommendations + case studies | applied |
| 13 | Efficiency (params, FLOPs, latency, ONNX) | Phase 4 |
| 14 | `REPORT.md` + zip | — |

### Positioning (literature check, October 2026)
- Accuracy collapse and overconfidence under controlled→field shift are **already documented** for generic crops (Xiang et al., *Front. Plant Sci.* 2026: PlantVillage→PlantDoc 99.7%→32%; Grad-CAM attention drifts to borders). A new 18-crop cross-region benchmark (Nalwanga et al., arXiv 2610.04456, Oct 2026) has **no onion** and no uncertainty, XAI or remedies.
- Conformal prediction for plant disease exists (MRC-Net, *Front. Plant Sci.* 2026), but it **explicitly omits cross-dataset shift**. Remedy systems (PlantCareNet 2025, LLM recommenders) **always** recommend and never withhold.
- In general ML, conformal sets over **action/label groups** exist (Conformal Structured Prediction, ICLR 2025; class-similarity CP with out-of-group penalty, PMLR 2026; Hierarchical Conformal Classification, arXiv 2508.13288), and so does **CP under multi-source shift** (arXiv 2602.14913; DS-CP, ICML 2026). Our RQ3 contribution is therefore **not a new CP algorithm**. It is the first use of these ideas for plant-disease *treatment decisions* under real cross-region shift, with a simple leave-one-source-out level calibration, evaluated by **wrong remedies issued**. These works must be cited.
- Fourier amplitude mixing + consistency (AmpMix, *Pattern Recognit.* 2023), CSRA and MixStyle (2021) are known. The new architectural element of SILA-Net is the **attention-consistency** term. So the architecture is a supporting contribution and is tested against **DG baselines and frozen foundation models**, not only plain CNNs.

### Contributions this notebook produces evidence for
1. **First multi-source onion benchmark across regions/sources** with a leakage audit (Roboflow augmented copies, cross-source and within-source duplicates, label-conflicting duplicates in TOM2024) and a reproducible group-aware protocol.
2. **Decision-level reliability (main contribution; applied, not a new CP algorithm).** *Remedy-level conformal prediction*: sets over **treatment groups** instead of diseases, so confusion between same-treatment diseases no longer blocks a safe recommendation. *Cross-source level calibration*: inner leave-one-source-out models measure how much conformal coverage drops on a never-seen region and set a stricter level α′, which mimics deployment in a new region. Measured as **wrong remedies issued vs acceptance rate**.
3. **SILA-Net**, a lightweight (4 M params) style-invariant lesion-attention model, compared against architectures, DG methods and foundation models, with ablations and phone-relevant latency.
4. **Quantitative XAI** (deletion/insertion, concentration, correct-vs-wrong, optional lesion masks) linking cross-source errors to where the model looks.

### Proposed model: **SILA-Net** (Style-Invariant Lesion Attention Network)
Lightweight EfficientNet-B0 backbone plus three parts that work together:
1. **Lesion attention head** (class-specific residual attention, CSRA-style). Every class gets its own spatial score map, so the model pools evidence from the most lesion-like locations instead of averaging the whole image. The maps are also *built-in explanations*.
2. **Style perturbation that keeps structure.** *Fourier low-frequency amplitude mixing* swaps illumination/colour/background "style" between images from different sources while keeping the phase (shape, lesion edges); *MixStyle* in the first 3 stages mixes feature statistics across sources.
3. **Style-invariance consistency.** Each image is seen as the original and as a Fourier-restyled view. The two views must agree on the **prediction** (symmetric KL) and on **where they look** (Jensen–Shannon between the true class's attention maps). Fourier mixing does not move pixels, so the attention maps should line up pixel for pixel. This pushes the model to rely on lesions, not on source-specific colour or background.

Every component is checked by ablation (section 6/9).

### How to run (read once)
1. **Settings (right panel):** Accelerator **GPU T4 x2 or P100** (one GPU is used); **Internet ON** (downloads data + pretrained weights). Your Kaggle account must be phone-verified for Internet.
2. **Add Input → Your Datasets → `tom2024-onion`** (TOM2024 cannot be downloaded from Kaggle servers).
3. **Add-ons → Secrets → `ROBOFLOW_API_KEY`**, tick *attach to this notebook*.
4. The rest is automatic: section 2 downloads COLD (Hugging Face) and both Roboflow sets, copies TOM2024, and saves everything as `onion_data.zip`. Section 2.0 stops right away with a clear message if any of 1–3 is missing.
5. **Recommended first:** set `SMOKE = True` → Run All (~15–20 min, tiny subset) to check everything on Kaggle. Then set `SMOKE = False`.
6. **Full run:** **Save Version → Save & Run All (Commit)**. This runs in the background (browser can be closed). The full plan (~230 runs, roughly **22–26 T4 GPU-hours**) is **more than one 12 h Kaggle session**. At `TIME_BUDGET_H` the notebook stops starting new runs, writes all analyses for what is finished, and saves everything.
7. **Resume:** in the next version, **Add Input → Notebook Output → (this notebook's previous version)**, then Save & Run All again. Data (`onion_data.zip`) and finished runs are reused automatically. Repeat until section 8 prints `ALL RUNS DONE`. Runs are ordered by priority, so the main tables fill first.

## 0. Configuration

In [ ]:
# ============================== CONFIG ==============================
SMOKE = False               # True = tiny subset, 1 epoch, 1 seed: checks the whole notebook in ~10-15 min
MERGE_VIRAL = True          # COLD Iris_yellow_virus + TOM2024 Virosis-D -> "viral" (check the grid in section 4)
IMG_SIZE = 224
CACHE_SIZE = 256            # every image is resized once to CACHE_SIZE x CACHE_SIZE (plain resize) and kept in RAM
EPOCHS = 15
BATCH = 64
SEEDS = [0, 1, 2]
ABLATION_SEEDS = [0]        # ablations: 1 seed by default (add 1, 2 if GPU quota allows: ~+6 GPU-hours)
CV_FOLDS = 5
VAL_FOLDS = 7               # inner validation = 1/7 of the training pool (groups kept intact)
DEDUP_HAMMING = 4           # pHash Hamming distance <= this -> near-duplicate
MIN_CLASS_IMAGES = 20       # harmonized classes with fewer images are dropped
ALPHA = 0.10                # conformal miscoverage -> 90% target coverage
TIME_BUDGET_H = 11.0        # no new training run is started after this many hours (Kaggle session = 12 h)
RUN_CV = True               # in-domain reference (needed for RQ1 "degradation")
RUN_ABLATIONS = True
RUN_FOUNDATION = True       # frozen foundation-model baselines (DINOv2, DINOv3, BioCLIP 2) + linear probe
RUN_DG_BASELINES = True     # domain-generalization baselines on the same EfficientNet-B0 (MixStyle, AmpMix, SWA)
RUN_CROSS_SOURCE_CAL = True # leave-one-SOURCE-out "inner" models -> cross-source conformal calibration
CROSS_CAL_MODELS = ["SILA-Net", "EfficientNetB0", "FM:DINOv2-B"]
SWA_START = 0.5             # SWA averages weights from this fraction of training onward
XAI_SAMPLES = 150          # per held-out dataset; also the size of the annotation pack
NUM_WORKERS = 8             # image loading threads
DATASETS = ["cold", "tom2024_onion", "roboflow_capstone", "roboflow_onionnet"]
REGION = {"cold": "India", "tom2024_onion": "Burkina Faso",
          "roboflow_capstone": "unknown", "roboflow_onionnet": "unknown"}

# SILA-Net hyper-parameters (fixed a priori; not tuned on held-out data)
CSRA_LAMBDA = 0.3           # weight of attention-pooled logits
FOURIER_BETA = 0.10         # half-size of the low-frequency window (fraction of H, W)
FOURIER_MAX_MIX = 1.0       # amplitude mixing ratio ~ U(0, FOURIER_MAX_MIX)
LAMBDA_KL = 1.0             # prediction consistency weight
LAMBDA_ATT = 1.0            # attention consistency weight
MIXSTYLE_P, MIXSTYLE_ALPHA = 0.5, 0.1

import os
from pathlib import Path
OUT = Path(os.environ.get("ONION_OUT", "/kaggle/working"))
INPUT_ROOT = Path(os.environ.get("ONION_INPUT", "/kaggle/input"))
CACHE_DIR = Path(os.environ.get("ONION_CACHE", "/tmp/onion_cache"))   # not saved as output
PRETRAINED = os.environ.get("ONION_PRETRAINED", "1") == "1"            # ImageNet weights (needs Internet)
RES = OUT / "results"
RUNS, WEIGHTS, FIGS, TABLES = RES / "runs", RES / "weights", RES / "figures", RES / "tables"
FEATS = RES / "features"    # cached foundation-model features
for p in (RUNS, WEIGHTS, FIGS, TABLES, FEATS, CACHE_DIR):
    p.mkdir(parents=True, exist_ok=True)

if SMOKE:
    EPOCHS, SEEDS, ABLATION_SEEDS, CV_FOLDS, XAI_SAMPLES = 1, [0], [0], 2, 8
    SMOKE_PER_CLASS = 24    # images per (dataset, class)
print("SMOKE" if SMOKE else "FULL", "run | output ->", RES)

## 0b. Package check (runs before any import)
Kaggle's preinstalled `timm` may be too old for DINOv3 (needs ≥ 1.0.20), and `open_clip` (BioCLIP 2) is not preinstalled. This cell upgrades only what is missing (needs Internet ON).

In [ ]:
import importlib.metadata as _md, subprocess, sys
def _ver(pkg):
    try:
        return tuple(int(x) for x in _md.version(pkg).split(".")[:3] if x.isdigit())
    except Exception:
        return None
_need = []
if (_ver("timm") or (0,)) < (1, 0, 20):
    _need.append("timm>=1.0.20")
if RUN_FOUNDATION and _ver("open_clip_torch") is None:
    _need.append("open_clip_torch")
if _need:
    print("installing:", _need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *_need], check=False)
print("timm", _md.version("timm") if _ver("timm") else "missing",
      "| open_clip", _md.version("open_clip_torch") if _ver("open_clip_torch") else "missing")

## 1. Imports and utilities

In [ ]:
import re, io, json, math, time, random, shutil, hashlib, zipfile, warnings
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy import stats
from scipy.fft import dct
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
T0 = time.time()
print("torch", torch.__version__, "| timm", timm.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if AMP else "")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def hours():
    return (time.time() - T0) / 3600

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))

def md_table(df, floatfmt="{:.3f}"):
    """DataFrame -> GitHub markdown (no extra dependency)."""
    df = df.reset_index() if not isinstance(df.index, pd.RangeIndex) else df
    cols = [str(c) for c in df.columns]
    fmt = lambda v: floatfmt.format(v) if isinstance(v, (float, np.floating)) and not pd.isna(v) else str(v)
    rows = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    rows += ["| " + " | ".join(fmt(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(rows)

REPORT = []   # (title, markdown) pairs collected for REPORT.md
try:
    from IPython.display import display, Markdown
    IN_NOTEBOOK = get_ipython() is not None
except Exception:
    IN_NOTEBOOK = False

def report(title, text):
    """Show a result block nicely in the notebook and keep it for REPORT.md."""
    REPORT.append((title, text))
    if IN_NOTEBOOK:
        display(Markdown(f"#### {title}\n\n{text}"))
    else:
        print(f"\n## {title}\n{text}")

## 1b. Resume: pick up runs from earlier versions of this notebook
Any attached input that contains `results/runs/*.json` (an earlier output of this notebook) is copied in, so finished runs are skipped.

In [ ]:
def resume_previous():
    n = 0
    if not INPUT_ROOT.exists():
        return 0
    for runs_dir in INPUT_ROOT.rglob("results/runs"):
        if runs_dir.resolve() == RUNS.resolve():
            continue
        for f in runs_dir.iterdir():
            if not (RUNS / f.name).exists():
                shutil.copy2(f, RUNS / f.name); n += 1
        for sub, dst in (("weights", WEIGHTS), ("features", FEATS)):
            sdir = runs_dir.parent / sub
            if sdir.exists():
                for f in sdir.iterdir():
                    if not (dst / f.name).exists():
                        shutil.copy2(f, dst / f.name)
        print("resumed from", runs_dir.parent)
    return n

print("files resumed:", resume_previous())

## 2. Get the data (reuse if attached, otherwise download)

| Source | Region | How it is obtained |
|---|---|---|
| COLD onion (raw split only, 815 images) | India | Hugging Face mirror `Project-AgML/COLD_onion_leaf_disease_classification`, `data_dir="raw"`. The augmented split is **never** loaded. |
| TOM2024 `onion_diseases` (Category A, 2,165 images) | Burkina Faso | your private Kaggle dataset **`tom2024-onion`** (Mendeley blocks Kaggle servers, so it has to be attached) |
| Onion Disease Capstone v2 | unknown | Roboflow API (YOLOv8 export) → every box cropped |
| OnionNet disease model 2, v10 | unknown | Roboflow API (YOLOv8 export) → every box cropped; **polygon labels are converted to boxes** (v2 skipped 1,421 of them) |

**Reuse first:** if the four prepared folders (or a zip of them) are already in `/kaggle/input` (e.g. `onion-disease-multi-source`, or the output of an earlier version of this notebook), nothing is downloaded. Fresh downloads go to local disk and are also saved as **one** file, `onion_data.zip`, in the output, so the next session (resume) reuses it.

### 2.0 Pre-flight check
Stops immediately with a clear message if something needed is missing, so no GPU time is wasted.

In [ ]:
import socket

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
DATA_LOCAL = CACHE_DIR / "data"           # downloaded / extracted data (local disk, not output)
RAW_YOLO = CACHE_DIR / "raw_yolo"

def find_dataset_dirs(root):
    """Find folders named like DATASETS (with class sub-folders) anywhere under `root`."""
    found = {}
    if not Path(root).exists():
        return found
    for r, dirs, _ in os.walk(root):
        for d in list(dirs):
            if d in DATASETS and d not in found:
                p = Path(r) / d
                if any(c.is_dir() for c in p.iterdir()):
                    found[d] = p
        dirs[:] = [d for d in dirs if d not in DATASETS and d != "results"]
        if len(found) == len(DATASETS):
            break
    return found

def find_data_zips(root):
    out = []
    for z in sorted(Path(root).rglob("*.zip")) if Path(root).exists() else []:
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()[:5000]
            if any(f"/{d}/" in "/" + n for n in names for d in DATASETS):
                out.append(z)
        except zipfile.BadZipFile:
            pass
    return out

def find_tom_dir():
    """TOM2024 Category A 'onion_diseases' folder anywhere under the inputs."""
    expected = {"alternaria_d", "healthy_leaf", "virosis-d", "fusarium-d", "bulb_blight-d"}
    for r, dirs, _ in os.walk(INPUT_ROOT):
        p = Path(r)
        if "onion-disease-multi-source" in p.as_posix() or "/results" in p.as_posix():
            continue
        if p.name.lower() == "onion_diseases" or len({d.lower() for d in dirs} & expected) >= 3:
            return p
    return None

def roboflow_key():
    key = os.environ.get("ROBOFLOW_API_KEY")
    if key:
        return key
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    except Exception:
        return None

def internet_ok(host="huggingface.co"):
    try:
        socket.create_connection((host, 443), timeout=5).close(); return True
    except OSError:
        return False

PREPARED = find_dataset_dirs(INPUT_ROOT)
PREPARED_ZIPS = find_data_zips(INPUT_ROOT) if len(PREPARED) < len(DATASETS) else []
need_download = len(PREPARED) < len(DATASETS) and not PREPARED_ZIPS
checks = {
    "GPU": (torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2 (or P100)"),
    "Internet": (internet_ok(), "Settings -> Internet -> On (pretrained weights; downloads)"),
}
if need_download:
    checks["TOM2024 dataset attached"] = (find_tom_dir() is not None, "Add Input -> Your Datasets -> tom2024-onion")
    checks["Roboflow API key"] = (roboflow_key() is not None,
                                  "Add-ons -> Secrets -> ROBOFLOW_API_KEY, tick 'attach to notebook'")
print("prepared data in inputs:", sorted(PREPARED) or "none", "| data zips:", [z.name for z in PREPARED_ZIPS] or "none")
print("data will be", "DOWNLOADED" if need_download else "REUSED")
bad = []
for name, (ok, fix) in checks.items():
    print(f"  [{'OK' if ok else 'MISSING'}] {name}" + ("" if ok else f"  ->  {fix}"))
    if not ok:
        bad.append(name)
if bad and not os.environ.get("ONION_ALLOW_PREFLIGHT_FAIL"):
    raise RuntimeError(f"Pre-flight failed: {bad}. Fix the items above, then Run All again.")

### 2.1 Reuse prepared data (folders or zip) if attached

In [ ]:
DATA_DIRS = dict(PREPARED)
if len(DATA_DIRS) < len(DATASETS) and PREPARED_ZIPS:
    for z in PREPARED_ZIPS:
        print("extracting", z, "->", DATA_LOCAL.parent)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(CACHE_DIR / "extracted")
    for d, p in find_dataset_dirs(CACHE_DIR / "extracted").items():
        DATA_DIRS.setdefault(d, p)
print("reused:", {d: str(p) for d, p in DATA_DIRS.items()} or "nothing")
DATA_DOWNLOADED = []

### 2.2 COLD (India): raw images only, original bytes

In [ ]:
def download_cold():
    from datasets import load_dataset, Image as HFImage
    src = os.environ.get("ONION_COLD_SRC", "Project-AgML/COLD_onion_leaf_disease_classification")
    cold = load_dataset(src, data_dir="raw", cache_dir=str(CACHE_DIR / "hf_cache"))
    out_root = DATA_LOCAL / "cold"
    shutil.rmtree(out_root, ignore_errors=True)
    ext_by_format = {"JPEG": ".jpg", "MPO": ".jpg", "PNG": ".png", "BMP": ".bmp", "WEBP": ".webp"}
    n = 0
    for split_name, split in cold.items():
        names = split.features["label"].names
        for i, row in enumerate(split.cast_column("image", HFImage(decode=False))):
            cls_dir = out_root / names[row["label"]].strip().replace(" ", "_")
            cls_dir.mkdir(parents=True, exist_ok=True)
            data = row["image"]["bytes"] or Path(row["image"]["path"]).read_bytes()
            with Image.open(io.BytesIO(data)) as im:
                if im.format in ext_by_format:                       # keep original bytes (no re-encoding)
                    (cls_dir / f"cold_{split_name}_{i:05d}{ext_by_format[im.format]}").write_bytes(data)
                else:
                    im.convert("RGB").save(cls_dir / f"cold_{split_name}_{i:05d}.png")
            n += 1
    shutil.rmtree(CACHE_DIR / "hf_cache", ignore_errors=True)
    assert 700 < n < 1000 or SMOKE or os.environ.get("ONION_COLD_SRC"), f"COLD raw should have ~815 images, got {n}"
    return out_root

if "cold" not in DATA_DIRS:
    try:
        import datasets  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
    DATA_DIRS["cold"] = download_cold(); DATA_DOWNLOADED.append("cold")
    print("COLD:", {d.name: len(list(d.iterdir())) for d in sorted(DATA_DIRS["cold"].iterdir())})

### 2.3 TOM2024 (Burkina Faso): copy from the attached Kaggle dataset

In [ ]:
if "tom2024_onion" not in DATA_DIRS:
    src = find_tom_dir()
    if src is None:
        raise RuntimeError("TOM2024 not found: Add Input -> Your Datasets -> tom2024-onion")
    out_root = DATA_LOCAL / "tom2024_onion"
    shutil.rmtree(out_root, ignore_errors=True)
    for cls_dir in sorted(d for d in src.iterdir() if d.is_dir()):
        (out_root / cls_dir.name).mkdir(parents=True, exist_ok=True)
        for f in sorted(cls_dir.rglob("*")):
            if f.suffix.lower() in IMG_EXTS:
                shutil.copy2(f, out_root / cls_dir.name / f.name)
    DATA_DIRS["tom2024_onion"] = out_root; DATA_DOWNLOADED.append("tom2024_onion")
    print("TOM2024 from", src, ":", {d.name: len(list(d.iterdir())) for d in sorted(out_root.iterdir())})

### 2.4 Roboflow sets: download (YOLOv8) and crop every annotation
Boxes (`cls cx cy w h`) are used directly; polygons (`cls x1 y1 x2 y2 ...`) become the box spanning their vertices. Each side is padded by 5% of the box size; crops under 32 px are skipped. File names keep the Roboflow stem (`<stem>__<split>__box<N>.jpg`), which section 2.7 uses to group crops by original photo.

In [ ]:
ROBOFLOW_SOURCES = [   # (workspace, project, version, dataset name)
    ("school-0xa1n", "onion-disease-capstone-final", 2, "roboflow_capstone"),
    ("onionnet", "onionnet_disease_model-2-hdkxc", 10, "roboflow_onionnet"),
]
PAD_PER_SIDE, MIN_CROP = 0.05, 32

def parse_label_line(line):
    parts = line.split()
    if len(parts) < 5:
        return None
    cls, vals = int(float(parts[0])), [float(v) for v in parts[1:]]
    if len(vals) == 4:
        cx, cy, w, h = vals
        return cls, "bbox", (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
    if len(vals) >= 6 and len(vals) % 2 == 0:
        xs, ys = vals[0::2], vals[1::2]
        return cls, "polygon", (min(xs), min(ys), max(xs), max(ys))
    return None

def yolo_to_crops(yolo_root, out_root):
    import yaml
    shutil.rmtree(out_root, ignore_errors=True)
    names = yaml.safe_load(open(yolo_root / "data.yaml"))["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    stats_ = Counter()
    for split in ["train", "valid", "test"]:
        img_dir, lbl_dir = yolo_root / split / "images", yolo_root / split / "labels"
        if not img_dir.exists():
            continue
        for img_path in sorted(img_dir.iterdir()):
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            lbl = lbl_dir / (img_path.stem + ".txt")
            lines = [l for l in lbl.read_text().splitlines() if l.strip()] if lbl.exists() else []
            if not lines:
                stats_["image_without_labels"] += 1; continue
            img = Image.open(img_path).convert("RGB"); W, H = img.size
            for j, line in enumerate(lines):
                p = parse_label_line(line)
                if p is None:
                    stats_["unparseable"] += 1; continue
                c, kind, (x1, y1, x2, y2) = p
                bw, bh = x2 - x1, y2 - y1
                px1, py1 = max(0, (x1 - PAD_PER_SIDE * bw) * W), max(0, (y1 - PAD_PER_SIDE * bh) * H)
                px2, py2 = min(W, (x2 + PAD_PER_SIDE * bw) * W), min(H, (y2 + PAD_PER_SIDE * bh) * H)
                if px2 - px1 < MIN_CROP or py2 - py1 < MIN_CROP:
                    stats_["too_small"] += 1; continue
                d = out_root / str(names[c]).strip().replace(" ", "_")
                d.mkdir(parents=True, exist_ok=True)
                img.crop((px1, py1, px2, py2)).save(d / f"{img_path.stem}__{split}__box{j}.jpg", quality=95)
                stats_[f"saved_{kind}"] += 1
    return dict(stats_)

todo_rf = [s for s in ROBOFLOW_SOURCES if s[3] not in DATA_DIRS]
if todo_rf:
    key = roboflow_key()
    for ws, proj, ver, name in todo_rf:
        target = RAW_YOLO / f"{name}_yolo"
        if not (target / "data.yaml").exists():
            if key is None:
                raise RuntimeError("ROBOFLOW_API_KEY missing (Add-ons -> Secrets).")
            try:
                from roboflow import Roboflow
            except ImportError:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=False)
                from roboflow import Roboflow
            Roboflow(api_key=key).workspace(ws).project(proj).version(ver).download("yolov8", location=str(target))
        st_ = yolo_to_crops(target, DATA_LOCAL / name)
        DATA_DIRS[name] = DATA_LOCAL / name; DATA_DOWNLOADED.append(name)
        print(f"{name}: {st_}")

### 2.5 Summary, and save downloaded data as one zip (reused by later sessions)

In [ ]:
def count_images(root):
    return {d.name: sum(1 for f in d.rglob("*") if f.suffix.lower() in IMG_EXTS)
            for d in sorted(p for p in Path(root).iterdir() if p.is_dir())}

rows = []
for d in DATASETS:
    if d in DATA_DIRS:
        for c, n in count_images(DATA_DIRS[d]).items():
            rows.append(dict(dataset=d, source_class=c, images=n))
src_tab = pd.DataFrame(rows)
src_tab.to_csv(TABLES / "source_class_counts.csv", index=False)
report("Source datasets (before any cleaning)",
       md_table(src_tab.pivot_table(index="source_class", columns="dataset", values="images", aggfunc="sum").fillna(0), "{:.0f}")
       + f"\n\nDownloaded this session: {DATA_DOWNLOADED or 'none (reused)'}")
assert len(DATA_DIRS) == len(DATASETS) or SMOKE, f"missing datasets: {[d for d in DATASETS if d not in DATA_DIRS]}"

if DATA_DOWNLOADED and not (OUT / "onion_data.zip").exists():
    with zipfile.ZipFile(OUT / "onion_data.zip", "w", zipfile.ZIP_STORED) as zf:
        for d, root in DATA_DIRS.items():
            for f in Path(root).rglob("*"):
                if f.is_file():
                    zf.write(f, f"data/{d}/{f.relative_to(root).as_posix()}")
    print("saved", OUT / "onion_data.zip", f"{(OUT / 'onion_data.zip').stat().st_size / 1e6:.0f} MB")
shutil.rmtree(RAW_YOLO, ignore_errors=True)

### 2.6 Roboflow augmented copies: why we group by original photo

Roboflow names files `IMG_0042_jpg.rf.<hash>`, and every augmented copy of `IMG_0042` gets a new hash; Roboflow augments only its *train* split. We therefore
- group crops by the **original photo** (`IMG_0042`), and
- keep **one copy per original photo** (preferring Roboflow `test`, then `valid`, i.e. never-augmented files). The number of copies removed is reported. This is the evidence for whether the export contained augmentation.

In [ ]:
RF_STEM = re.compile(r"^(?P<orig>.+?)_(?:jpe?g|png|bmp|webp)\.rf\.[0-9a-z]+$", re.IGNORECASE)
for d in DATASETS:
    print(f"{d:20s} -> {DATA_DIRS.get(d, 'NOT FOUND')}")

### 2.7 Index every image (dataset, class, original photo, Roboflow split)

In [ ]:
rows = []
for ds, root in DATA_DIRS.items():
    for cls_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for f in sorted(cls_dir.rglob("*")):
            if f.suffix.lower() not in IMG_EXTS:
                continue
            r = dict(path=str(f), dataset=ds, source_class=cls_dir.name, rel=f"{ds}/{cls_dir.name}/{f.name}")
            if ds.startswith("roboflow"):
                parts = f.stem.split("__")
                r["rf_stem"] = parts[0]
                r["rf_split"] = parts[1] if len(parts) > 1 else "unknown"
                m = RF_STEM.match(parts[0])
                r["orig"] = m.group("orig") if m else parts[0]
            else:
                r["rf_stem"], r["rf_split"], r["orig"] = f.stem, "", f.stem
            rows.append(r)
idx_all = pd.DataFrame(rows)
idx_all["group"] = idx_all.dataset + "/" + idx_all.orig
print(idx_all.groupby("dataset").size().to_string())

### 2.3 Roboflow audit: remove augmented copies, keep distinct photos

Roboflow augments **only the train split**; `valid`/`test` files are never augmented. Rules:
- every `valid`/`test` file is kept (two such files with the same original name are *different photos* that share a filename);
- in `train`, the copies of one photo have the **same box labels**, so per original name we keep **one file per label signature** (sorted list of its box classes).

Splitting groups stay at the **original-name** level (conservative: name-sharing photos always land on the same side).

In [ ]:
rf = idx_all[idx_all.dataset.str.startswith("roboflow")].copy()
is_rf = idx_all.dataset.str.startswith("roboflow")
if len(rf):
    sig = rf.groupby(["dataset", "rf_stem"]).source_class.agg(lambda s: "|".join(sorted(s)))
    stems = rf.drop_duplicates(["dataset", "rf_stem"])[["dataset", "rf_stem", "rf_split", "orig"]].copy()
    stems["sig"] = sig.reindex(pd.MultiIndex.from_arrays([stems.dataset, stems.rf_stem])).values
    stems = stems.sort_values(["dataset", "orig", "rf_stem"])
    is_train = stems.rf_split.eq("train")
    keep_stems = pd.concat([stems[~is_train],
                            stems[is_train].drop_duplicates(["dataset", "orig", "sig"])])
    keep_set = set(zip(keep_stems.dataset, keep_stems.rf_stem))
    keep_canon = ~is_rf | pd.Series([(d, s) in keep_set for d, s in zip(idx_all.dataset, idx_all.rf_stem)])
    txt = []
    for ds in sorted(stems.dataset.unique()):
        st_ = stems[stems.dataset == ds]
        tr_cp = st_[st_.rf_split == "train"].groupby(["orig", "sig"]).size()
        ev = st_[st_.rf_split != "train"].groupby("orig").size()
        n_before = int((is_rf & (idx_all.dataset == ds)).sum()); n_after = int((keep_canon & (idx_all.dataset == ds)).sum())
        txt.append(f"- **{ds}**: exported files per split {st_.rf_split.value_counts().to_dict()}; "
                   f"train copies per (photo, label set) {dict(sorted(Counter(tr_cp).items()))} "
                   f"(a dominant 3 = Roboflow 3x augmentation); valid/test files per name {dict(sorted(Counter(ev).items()))}; "
                   f"crops {n_before} -> {n_after}")
    report("Roboflow augmentation audit", "\n".join(txt))

    # visual check: the photo names with the most exported files (are they copies, or different photos?)
    many = stems.groupby(["dataset", "orig"]).size().sort_values(ascending=False).head(4)
    fig, axes = plt.subplots(len(many), 6, figsize=(13, 2.3 * len(many)), squeeze=False)
    for r, ((ds, orig), n) in enumerate(many.items()):
        files = idx_all[(idx_all.dataset == ds) & (idx_all.orig == orig)].drop_duplicates("rf_stem").head(6)
        for c in range(6):
            ax = axes[r, c]; ax.axis("off")
            if c < len(files):
                fr = files.iloc[c]
                ax.imshow(Image.open(fr.path).convert("RGB"))
                ax.set_title(f"{fr.rf_split} | {fr.source_class[:14]}", fontsize=7)
        axes[r, 0].text(-0.05, 0.5, f"{ds}\n{orig[:18]}\n({n} files)", transform=axes[r, 0].transAxes,
                        ha="right", va="center", fontsize=8)
    plt.tight_layout(); plt.savefig(FIGS / "roboflow_copies_check.png", dpi=100); plt.show()
else:
    keep_canon = pd.Series(True, index=idx_all.index)
idx = idx_all[keep_canon.values].reset_index(drop=True)

if SMOKE:
    idx = (idx.sample(frac=1.0, random_state=0).groupby(["dataset", "source_class"]).head(SMOKE_PER_CLASS)
              .sort_values("rel").reset_index(drop=True))
print("images after canonicalization:", len(idx))

## 3. Load images once (RAM cache) and deduplicate

1. Every image is EXIF-corrected, converted to RGB and resized to `CACHE_SIZE`². All models train from this single cache, so preprocessing is identical for every model.
2. **Exact duplicates** (same file bytes): one copy is kept; if the copies carry *different* labels, all of them are dropped.
3. **Near-duplicates** (DCT perceptual hash, Hamming ≤ `DEDUP_HAMMING`), within and across datasets, are joined into clusters (union–find):
   - cluster with conflicting harmonized labels → dropped,
   - cluster spanning datasets → kept only in the dataset holding most of the cluster (so a held-out set never shares an image with training),
   - otherwise → all members get the **same group**, so they always land on the same side of a split.

### 3.0 Image loader

In [ ]:
def load_one(path):
    raw = Path(path).read_bytes()
    with Image.open(io.BytesIO(raw)) as im:
        im = ImageOps.exif_transpose(im).convert("RGB").resize((CACHE_SIZE, CACHE_SIZE), Image.BILINEAR)
        arr = np.asarray(im, dtype=np.uint8).transpose(2, 0, 1)
    return arr, hashlib.md5(raw).hexdigest()

### 3.1 Load all images into the RAM cache (runs once, then reused)

In [ ]:
cache_file = CACHE_DIR / f"cache_{CACHE_SIZE}_{len(idx)}_{hashlib.md5(''.join(idx.rel).encode()).hexdigest()[:10]}.npy"
md5_file = cache_file.with_suffix(".md5.json")
if cache_file.exists() and md5_file.exists():
    CACHE = np.load(cache_file, mmap_mode=None)
    idx["md5"] = json.loads(md5_file.read_text())
else:
    CACHE = np.empty((len(idx), 3, CACHE_SIZE, CACHE_SIZE), dtype=np.uint8)
    md5s = [None] * len(idx)
    def _work(i):
        CACHE[i], md5s[i] = load_one(idx.path[i])
    with ThreadPoolExecutor(NUM_WORKERS) as ex:
        list(tqdm(ex.map(_work, range(len(idx))), total=len(idx), desc="loading"))
    idx["md5"] = md5s
    np.save(cache_file, CACHE); md5_file.write_text(json.dumps(md5s))
print("cache:", CACHE.shape, f"{CACHE.nbytes / 1e9:.2f} GB")

### 3b. Harmonized labels (needed by the dedup conflict rule)

In [ ]:
LABEL_MAP = {
    "cold": {"Iris_yellow_virus": "viral",
             "Stemphylium_leaf_blight_and_collectrichum_leaf_blight": "stemphylium",
             "healthy": "healthy", "purple_blotch": "purple_blotch"},
    "tom2024_onion": {"Alternaria_D": "purple_blotch", "Bulb_blight-D": "bulb_blight",
                      "Fusarium-D": "fusarium", "Healthy_leaf": "healthy", "Virosis-D": "viral"},
    "roboflow_capstone": {"Botrytis_Leaf_Blight": "botrytis", "Downy_Mildew": "downy_mildew",
                          "Healthy": "healthy", "Purple_Blotch": "purple_blotch", "Rust": "rust",
                          "Stemphylium_Leaf_Blight": "stemphylium",
                          "Xanthomonas_Leaf_Blight": "xanthomonas"},
    "roboflow_onionnet": {"Burn_Tip": "burn_tip", "Downy_Mildew": "downy_mildew",
                          "Healthy": "healthy", "Purple_Blotch": "purple_blotch"},
}
if not MERGE_VIRAL:
    LABEL_MAP["cold"]["Iris_yellow_virus"] = "iris_yellow_virus"
    LABEL_MAP["tom2024_onion"]["Virosis-D"] = "virosis"

idx["label_name"] = [LABEL_MAP.get(d, {}).get(c) for d, c in zip(idx.dataset, idx.source_class)]
unmapped = idx[idx.label_name.isna()].groupby(["dataset", "source_class"]).size()
if len(unmapped):
    print("WARNING unmapped classes (dropped):\n", unmapped)
idx["drop_reason"] = np.where(idx.label_name.isna(), "unmapped_class", "")

### 3.3 Exact duplicates (identical file bytes)

In [ ]:
# ---------- exact duplicates
dup_txt = []
g = idx[idx.drop_reason == ""].groupby("md5")
conflict_md5 = {m for m, s in g.label_name.nunique().items() if s > 1}
idx.loc[idx.md5.isin(conflict_md5), "drop_reason"] = "exact_dup_label_conflict"
dupmask = idx.drop_reason.eq("") & idx.duplicated("md5", keep="first")
idx.loc[dupmask, "drop_reason"] = "exact_dup_extra_copy"
dup_txt.append(f"- exact duplicates: {len(conflict_md5)} sets with conflicting labels (all copies dropped), "
               f"{int(dupmask.sum())} extra copies dropped")
for ds, n in idx[idx.drop_reason.str.startswith("exact")].groupby("dataset").size().items():
    dup_txt.append(f"  - {ds}: {n} images dropped")

### 3.4 Perceptual hash (pHash) of every image

In [ ]:
# ---------- perceptual hash (DCT pHash, 64 bit) from the cache
def phash_batch(arr):
    """arr: N x 3 x S x S uint8 -> N uint64 pHash (32x32 grayscale DCT, 8x8 low band vs median)."""
    t = torch.from_numpy(arr).float()
    gray = (0.299 * t[:, 0] + 0.587 * t[:, 1] + 0.114 * t[:, 2]).unsqueeze(1)
    small = F.interpolate(gray, size=(32, 32), mode="area").squeeze(1).numpy()
    d = dct(dct(small, axis=1, norm="ortho"), axis=2, norm="ortho")[:, :8, :8].reshape(len(arr), 64)
    bits = d > np.median(d, axis=1, keepdims=True)
    return np.packbits(bits, axis=1).view(">u8").ravel().astype(np.uint64)

alive = np.where(idx.drop_reason.eq(""))[0]
H = np.concatenate([phash_batch(CACHE[alive[i:i + 2048]]) for i in range(0, len(alive), 2048)])
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

### 3.5 Find near-duplicate pairs and build clusters (union–find)

In [ ]:
parent = np.arange(len(alive))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
Hb = H.view(np.uint8).reshape(-1, 8)
n_pairs = 0
for s in tqdm(range(0, len(H), 512), desc="pHash pairs"):
    x = np.bitwise_xor(Hb[s:s + 512, None, :], Hb[None, :, :])        # B x N x 8
    dist = POP[x].sum(-1, dtype=np.uint16)
    ii, jj = np.nonzero(dist <= DEDUP_HAMMING)
    for i, j in zip(ii + s, jj):
        if i < j:
            ri, rj = find(i), find(j)
            if ri != rj:
                parent[rj] = ri
            n_pairs += 1
roots = np.array([find(i) for i in range(len(alive))])
sub = idx.loc[alive, ["dataset", "label_name", "group"]].copy()
sub["cluster"] = roots
csize = sub.cluster.map(sub.cluster.value_counts())
multi = sub[csize > 1]
dup_txt.append(f"- near-duplicate pairs (Hamming <= {DEDUP_HAMMING}): {n_pairs}; clusters with >1 image: {multi.cluster.nunique()}")

### 3.6 Apply the dedup rules (label conflicts, cross-dataset copies, group merging)

In [ ]:
# conflicting labels -> drop cluster
lab_conf = multi.groupby("cluster").label_name.nunique()
bad = set(lab_conf[lab_conf > 1].index)
idx.loc[sub.index[sub.cluster.isin(bad)], "drop_reason"] = "near_dup_label_conflict"
dup_txt.append(f"- near-duplicate clusters with conflicting labels: {len(bad)} "
               f"({int(sub.cluster.isin(bad).sum())} images dropped)")

# cross-dataset -> keep only in the majority dataset
cross = multi[~multi.cluster.isin(bad)].groupby("cluster").dataset.nunique()
n_cross_drop = 0
for c in cross[cross > 1].index:
    members = sub[sub.cluster == c]
    keep_ds = members.dataset.value_counts().index[0]
    drop_ix = members.index[members.dataset != keep_ds]
    idx.loc[drop_ix, "drop_reason"] = "near_dup_cross_dataset"
    n_cross_drop += len(drop_ix)
dup_txt.append(f"- clusters spanning datasets: {int((cross > 1).sum())} ({n_cross_drop} images dropped from the minority dataset)")
pairs_by_ds = multi.groupby("cluster").dataset.agg(lambda s: "+".join(sorted(set(s))))
for k, v in pairs_by_ds[pairs_by_ds.str.contains(r"\+")].value_counts().items():
    dup_txt.append(f"  - {k}: {v} clusters")

# within dataset -> merge groups: union-find over GROUPS, so every crop of every photo in a cluster moves together
gparent = {}
def gfind(a):
    gparent.setdefault(a, a)
    while gparent[a] != a:
        gparent[a] = gparent[gparent[a]]; a = gparent[a]
    return a
still_kept = idx.loc[sub.index, "drop_reason"].eq("").values
for _, members in sub[(csize > 1).values & still_kept].groupby("cluster").group:
    gs = sorted(set(members))
    for other in gs[1:]:
        ra, rb = gfind(gs[0]), gfind(other)
        if ra != rb:
            gparent[rb] = ra
n_merged = sum(1 for g in list(gparent) if gfind(g) != g)
idx["group"] = [("merged/" + gfind(g)) if g in gparent else g for g in idx.group]
dup_txt.append(f"- photo groups merged because of near-duplicates: {n_merged}")
report("Deduplication", "\n".join(dup_txt))

## 4. Final label set and sanity grid

Classes with fewer than `MIN_CLASS_IMAGES` images are dropped. **Look at the grid**: every row should show the same disease across datasets. If *Iris yellow virus* (COLD) and *Virosis-D* (TOM2024) look like different symptoms, set `MERGE_VIRAL = False`.

In [ ]:
counts = idx[idx.drop_reason == ""].label_name.value_counts()
small = set(counts[counts < MIN_CLASS_IMAGES].index)
idx.loc[idx.label_name.isin(small) & idx.drop_reason.eq(""), "drop_reason"] = "class_too_small"
meta = idx[idx.drop_reason == ""].copy()
CLASSES = sorted(meta.label_name.unique())
C2I = {c: i for i, c in enumerate(CLASSES)}
idx["y"] = idx.label_name.map(C2I).fillna(-1).astype(int)
meta["y"] = meta.label_name.map(C2I)
DS2I = {d: i for i, d in enumerate(DATASETS)}
Y_ALL = torch.from_numpy(idx.y.values)
D_ALL = torch.from_numpy(idx.dataset.map(DS2I).values)

tab = pd.crosstab(meta.label_name, meta.dataset).reindex(columns=[d for d in DATASETS if d in DATA_DIRS])
tab["total"] = tab.sum(axis=1)
tab["n_sources"] = (tab.drop(columns="total") > 0).sum(axis=1)
report("Harmonized class counts (after dedup)", md_table(tab, "{:.0f}") +
       f"\n\nDropped images by reason: {idx.drop_reason.replace('', 'kept').value_counts().to_dict()}")
idx.to_csv(TABLES / "manifest_dedup.csv", index=False)

### 4.1 Sanity grid: does each harmonized class look the same in every dataset?

In [ ]:
# sanity grid: rows = classes present in >1 dataset, columns = up to 3 samples per dataset
shared = [c for c in CLASSES if tab.loc[c, "n_sources"] > 1]
dss = [d for d in DATASETS if d in DATA_DIRS]
fig, axes = plt.subplots(len(shared), 3 * len(dss), figsize=(2 * 3 * len(dss), 2.1 * len(shared)), squeeze=False)
rng = np.random.default_rng(0)
for r, c in enumerate(shared):
    for k, d in enumerate(dss):
        pool = meta.index[(meta.label_name == c) & (meta.dataset == d)].values
        pick = rng.choice(pool, min(3, len(pool)), replace=False) if len(pool) else []
        for j in range(3):
            ax = axes[r, 3 * k + j]; ax.axis("off")
            if j < len(pick):
                ax.imshow(CACHE[pick[j]].transpose(1, 2, 0))
            if r == 0 and j == 0:
                ax.set_title(d, fontsize=9, loc="left")
        axes[r, 0].text(-0.1, 0.5, c, transform=axes[r, 0].transAxes, ha="right", va="center", fontsize=10)
plt.tight_layout(); plt.savefig(FIGS / "label_harmonization_grid.png", dpi=110); plt.show()

### 4.2 *Iris yellow virus* (COLD) vs *Virosis-D* (TOM2024): decide `MERGE_VIRAL`

In [ ]:
vir = [("cold", "Iris_yellow_virus"), ("tom2024_onion", "Virosis-D")]
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for r, (d, sc) in enumerate(vir):
    pool = idx.index[(idx.dataset == d) & (idx.source_class == sc)].values
    pick = rng.choice(pool, min(8, len(pool)), replace=False) if len(pool) else []
    for j in range(8):
        axes[r, j].axis("off")
        if j < len(pick):
            axes[r, j].imshow(CACHE[pick[j]].transpose(1, 2, 0))
    axes[r, 0].set_title(f"{d}: {sc}", loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(FIGS / "viral_iyv_vs_virosis.png", dpi=110); plt.show()
print("figures:", FIGS / "label_harmonization_grid.png", FIGS / "viral_iyv_vs_virosis.png")

## 5. Splits: LODO + stratified group 5-fold, with leakage checks

- **LODO** (RQ1/RQ2): train on 3 datasets, test on the 4th. Inner validation (early stopping, temperature scaling, conformal calibration) = one group-stratified fold of the training pool. The held-out set is split into **ID** (classes seen in training; used for accuracy/F1) and **unseen-class** images (the model cannot know them; used to test whether gating withholds them).
- **CV** (in-domain reference): stratified (dataset × class) group 5-fold over all four datasets. In-domain F1 for dataset *D* is computed on its own test folds, restricted to the same classes as LODO-*D*, so **degradation = in-domain - LODO** compares like with like.
- Assertions: no group is in two partitions, and no image id repeats.

In [ ]:
def strat_key(df):
    return (df.dataset + "|" + df.label_name).astype("category").cat.codes.values

def inner_split(pool_ix, seed=0):
    df = meta.loc[pool_ix]
    sgk = StratifiedGroupKFold(n_splits=VAL_FOLDS, shuffle=True, random_state=seed)
    tr, va = next(sgk.split(df, strat_key(df), df.group))
    return df.index.values[tr], df.index.values[va]

def check_disjoint(*parts):
    for a in range(len(parts)):
        for b in range(a + 1, len(parts)):
            ga, gb = set(idx.group[parts[a]]), set(idx.group[parts[b]])
            assert not (ga & gb), f"GROUP LEAK between partitions {a} and {b}: {list(ga & gb)[:3]}"
            assert not (set(parts[a]) & set(parts[b])), "index overlap"

### 5.1 Leave-one-dataset-out (LODO) splits

In [ ]:
SPLITS = {}
for held in dss:
    pool = meta.index[meta.dataset != held].values
    test = meta.index[meta.dataset == held].values
    tr, va = inner_split(pool)
    train_classes = set(meta.y[tr])
    va = va[np.isin(meta.y[va], list(train_classes))]
    te_id = test[np.isin(meta.y[test], list(train_classes))]
    te_ood = test[~np.isin(meta.y[test], list(train_classes))]
    check_disjoint(tr, va, test)
    SPLITS[("lodo", held)] = dict(train=tr, val=va, test=test, test_id=te_id, test_ood=te_ood)

### 5.1b Inner leave-one-*source*-out splits (for cross-source conformal calibration)

For held-out set *H*, the training sources are the other three. For each training source *S*, an **inner model** is trained on the remaining two sources and predicts *S*, which it has never seen. This measures how much standard conformal coverage drops on **a region the model was not trained on**, which is exactly the deployment situation. From it we learn how much stricter the conformal level must be (α′ < α) to keep 90% coverage. The main model then uses α′ on its own validation scores (section 10). *H* is never touched.

In [ ]:
for held in dss:
    sources = [d for d in dss if d != held]
    for src in sources:
        pool = meta.index[meta.dataset.isin([d for d in sources if d != src])].values
        test = meta.index[meta.dataset == src].values
        tr, va = inner_split(pool)
        train_classes = set(meta.y[tr])
        va = va[np.isin(meta.y[va], list(train_classes))]
        te_id = test[np.isin(meta.y[test], list(train_classes))]
        assert not set(meta.loc[np.concatenate([tr, va, test])].dataset) & {held}, "held-out set leaked into inner split"
        check_disjoint(tr, va, test)
        SPLITS[("inner", f"{held}|{src}")] = dict(train=tr, val=va, test=test, test_id=te_id,
                                                   test_ood=test[~np.isin(meta.y[test], list(train_classes))])

### 5.2 Stratified group 5-fold CV (in-domain reference)

In [ ]:
sgk = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
for k, (pool_i, te_i) in enumerate(sgk.split(meta, strat_key(meta), meta.group)):
    pool, test = meta.index.values[pool_i], meta.index.values[te_i]
    tr, va = inner_split(pool, seed=k)
    va = va[np.isin(meta.y[va], list(set(meta.y[tr])))]
    check_disjoint(tr, va, test)
    SPLITS[("cv", f"fold{k}")] = dict(train=tr, val=va, test=test, test_id=test, test_ood=test[:0])

### 5.3 Split summary

In [ ]:
rows = []
for (proto, name), s in SPLITS.items():
    rows.append(dict(protocol=proto, split=name, train=len(s["train"]), val=len(s["val"]),
                     test=len(s["test"]), test_id=len(s["test_id"]), test_unseen_class=len(s["test_ood"]),
                     train_classes=len(set(meta.y[s["train"]])),
                     shared_test_classes=", ".join(CLASSES[c] for c in sorted(set(meta.y[s["test_id"]])))))
split_df = pd.DataFrame(rows)
split_df.to_csv(TABLES / "splits_summary.csv", index=False)
report("Splits (all leakage checks passed)", md_table(split_df, "{:.0f}"))

### 5.4 Save split signature and per-image split membership (supplementary material)

In [ ]:
split_sig = hashlib.md5(json.dumps({f"{k[0]}/{k[1]}": sorted(map(int, v["test"])) for k, v in SPLITS.items()}).encode()).hexdigest()
# a run is only reused if it was trained on exactly these splits with the same training budget (never smoke runs)
RUN_TAG = hashlib.md5(f"{split_sig}|{EPOCHS}|{IMG_SIZE}|{SMOKE}".encode()).hexdigest()[:12]
sig_file = RES / "split_signature.txt"
if sig_file.exists() and sig_file.read_text() != split_sig:
    print("WARNING: splits differ from the resumed runs (data or config changed). Old runs are kept but may not be comparable.")
sig_file.write_text(split_sig)
# per-image split membership, for the paper's supplementary material
mem = pd.DataFrame(index=meta.index)
for (proto, name), s in SPLITS.items():
    col = pd.Series("", index=meta.index)
    for part in ("train", "val", "test"):
        col[s[part]] = part
    mem[f"{proto}_{name}"] = col
pd.concat([meta[["rel", "dataset", "label_name", "group"]], mem], axis=1).to_csv(TABLES / "split_membership.csv")

## 6. Models

Three families of baselines, so the comparison covers what a 2026 reviewer expects:

| Family | Models | Why |
|---|---|---|
| **Architectures** (fine-tuned, ImageNet weights) | ResNet-50, EfficientNet-B0, MobileNetV3-L, ConvNeXt-T, ViT-S | standard CNN/ViT references |
| **Foundation models** (frozen + linear probe) | DINOv2-B/14, DINOv3-B/16, BioCLIP 2 (ViT-L/14) | strongest under field shift in recent benchmarks (Xiang et al. 2026; frozen FM fusion 2026) |
| **Domain generalization** (same EfficientNet-B0 as SILA-Net) | MixStyle, AmpMix + consistency (Fourier, FACT/AmpMix-style), SWA | isolates whether SILA-Net beats *known DG recipes*, not just plain training |

**Proposed:** SILA-Net (EfficientNet-B0 + lesion attention head + Fourier style view + MixStyle + prediction & attention consistency).
**Ablations** (same backbone; each adds one part): `+CSRA` → `+Fourier view` → `+MixStyle` → `+pred. consistency` → full SILA-Net (adds attention consistency); plus `SILA w/o CSRA` (GAP head).

All trained models share the same data, sampler, geometric/colour augmentation, optimizer, schedule and epochs. Foundation models see exactly the same images (evaluation view) and the same class-balanced loss weights.

In [ ]:
BACKBONES = {
    "resnet50":        dict(timm="resnet50.a1_in1k", lr=3e-4),
    "efficientnet_b0": dict(timm="efficientnet_b0.ra_in1k", lr=3e-4),
    "mobilenetv3_l":   dict(timm="mobilenetv3_large_100.ra_in1k", lr=3e-4),
    "convnext_tiny":   dict(timm="convnext_tiny.fb_in1k", lr=2e-4),
    "vit_small":       dict(timm="vit_small_patch16_224.augreg_in21k_ft_in1k", lr=1e-4),
}
MAIN_MODELS = {
    "ResNet50":        dict(backbone="resnet50", head="timm"),
    "EfficientNetB0":  dict(backbone="efficientnet_b0", head="timm"),
    "MobileNetV3L":    dict(backbone="mobilenetv3_l", head="timm"),
    "ConvNeXtT":       dict(backbone="convnext_tiny", head="timm"),
    "ViTS":            dict(backbone="vit_small", head="timm"),
}
FOUNDATION_MODELS = {   # frozen encoder + linear probe (head="probe")
    "FM:DINOv2-B":  dict(head="probe", kind="timm", name="vit_base_patch14_dinov2.lvd142m"),
    "FM:DINOv3-B":  dict(head="probe", kind="timm", name="vit_base_patch16_dinov3.lvd1689m"),
    "FM:BioCLIP-2": dict(head="probe", kind="open_clip", name="hf-hub:imageomics/bioclip-2"),
}
DG_MODELS = {
    "DG:MixStyle":      dict(backbone="efficientnet_b0", head="timm", mixstyle=True),
    "DG:AmpMix+Cons":   dict(backbone="efficientnet_b0", head="timm", fourier=True, cons_pred=True),
    "DG:SWA":           dict(backbone="efficientnet_b0", head="timm", swa=True),
}
if RUN_FOUNDATION:
    MAIN_MODELS.update(FOUNDATION_MODELS)
if RUN_DG_BASELINES:
    MAIN_MODELS.update(DG_MODELS)
MAIN_MODELS["SILA-Net"] = dict(backbone="efficientnet_b0", head="csra", fourier=True, mixstyle=True,
                               cons_pred=True, cons_att=True)
CV_MODELS = [m for m in MAIN_MODELS if m not in DG_MODELS]   # in-domain reference (RQ1)
ABLATION_MODELS = {
    "Abl:+CSRA":               dict(backbone="efficientnet_b0", head="csra"),
    "Abl:+CSRA+Fourier":       dict(backbone="efficientnet_b0", head="csra", fourier=True),
    "Abl:+CSRA+Fourier+MixStyle": dict(backbone="efficientnet_b0", head="csra", fourier=True, mixstyle=True),
    "Abl:+...+PredCons":       dict(backbone="efficientnet_b0", head="csra", fourier=True, mixstyle=True, cons_pred=True),
    "Abl:SILA w/o CSRA":       dict(backbone="efficientnet_b0", head="timm", fourier=True, mixstyle=True, cons_pred=True),
}
ALL_MODELS = {**MAIN_MODELS, **ABLATION_MODELS}
PROPOSED = "SILA-Net"
ABL_CHAIN = ["EfficientNetB0", "Abl:+CSRA", "Abl:+CSRA+Fourier", "Abl:+CSRA+Fourier+MixStyle",
             "Abl:+...+PredCons", "SILA-Net", "Abl:SILA w/o CSRA"]

### 6.1 MixStyle (feature-statistics mixing across sources)

In [ ]:
def mixstyle(x, p=MIXSTYLE_P, alpha=MIXSTYLE_ALPHA, eps=1e-6):
    """MixStyle (Zhou et al., ICLR 2021): mix channel-wise feature statistics between random pairs."""
    if x.dim() != 4 or random.random() > p:
        return x
    dt, x = x.dtype, x.float()
    B = x.size(0)
    mu = x.mean((2, 3), keepdim=True)
    sig = (x.var((2, 3), keepdim=True) + eps).sqrt()
    mu, sig = mu.detach(), sig.detach()
    lam = torch.distributions.Beta(alpha, alpha).sample((B, 1, 1, 1)).to(x.device)
    perm = torch.randperm(B, device=x.device)
    mu2, sig2 = lam * mu + (1 - lam) * mu[perm], lam * sig + (1 - lam) * sig[perm]
    return (((x - mu) / sig) * sig2 + mu2).to(dt)

### 6.2 The network: backbone + timm head (baselines) or lesion-attention head (SILA-Net)

In [ ]:
def stage_modules(bb):
    if hasattr(bb, "layer1"):
        return [bb.layer1, bb.layer2, bb.layer3]
    if hasattr(bb, "stages"):
        return list(bb.stages)[:3]
    if hasattr(bb, "blocks") and isinstance(bb.blocks, nn.Sequential):
        return list(bb.blocks)[:3]
    return []

class Net(nn.Module):
    def __init__(self, spec, n_classes, pretrained=True):
        super().__init__()
        self.spec = spec
        name = BACKBONES[spec["backbone"]]["timm"]
        if spec["head"] == "csra":
            self.backbone = timm.create_model(name, pretrained=pretrained and PRETRAINED, num_classes=0)
            self.drop = nn.Dropout(0.2)
            self.cls = nn.Conv2d(self.backbone.num_features, n_classes, 1)
        else:
            self.backbone = timm.create_model(name, pretrained=pretrained and PRETRAINED, num_classes=n_classes, drop_rate=0.2)
        cfg = timm.data.resolve_data_config({}, model=self.backbone)
        self.register_buffer("mean", torch.tensor(cfg["mean"]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(cfg["std"]).view(1, 3, 1, 1))
        self.mix_on = False
        if spec.get("mixstyle"):
            for m in stage_modules(self.backbone):
                m.register_forward_hook(lambda mod, inp, out: mixstyle(out) if (self.training and self.mix_on) else out)

    def head_params(self):
        return list(self.cls.parameters()) if self.spec["head"] == "csra" else list(self.backbone.get_classifier().parameters())

    def features(self, x01):
        """x01: images in [0, 1]."""
        return self.backbone.forward_features((x01 - self.mean) / self.std)

    def head(self, f):
        if self.spec["head"] == "csra":
            S = self.cls(self.drop(f))                      # B x K x h x w class score maps
            flat = S.flatten(2)
            att = flat.float().softmax(-1)
            logits = flat.float().mean(-1) + CSRA_LAMBDA * (att * flat.float()).sum(-1)
            return logits, att.view(S.shape)
        return self.backbone.forward_head(f), None

    def forward(self, x01):
        return self.head(self.features(x01))

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

## 7. Training engine

- Images stay in RAM as uint8; augmentation runs **on the GPU**: random-resized crop + rotation ±15° + horizontal flip (one affine warp), mild brightness/contrast/saturation jitter. **No hue jitter**, because colour (purple vs yellow) is diagnostic.
- **Sampler** weights ∝ (class count)^-½ × (source count)^-½: helps the rare classes and keeps sources balanced. Same sampler for every model.
- AdamW (head lr ×10), 1-epoch warm-up + cosine, label smoothing 0.1, AMP. The checkpoint with the best **validation macro-F1** (source validation only, never the held-out set) is kept.

In [ ]:
def gpu_augment(xb):
    """xb: uint8 B x 3 x S x S on device -> float [0,1] B x 3 x IMG x IMG."""
    x = xb.float() / 255.0
    B, dev = x.size(0), x.device
    area = torch.empty(B, device=dev).uniform_(0.55, 1.0)
    logr = torch.empty(B, device=dev).uniform_(math.log(3 / 4), math.log(4 / 3))
    sw = torch.sqrt(area * torch.exp(logr)).clamp(max=1.0)
    sh = torch.sqrt(area / torch.exp(logr)).clamp(max=1.0)
    tx = (1 - sw) * torch.empty(B, device=dev).uniform_(-1, 1)
    ty = (1 - sh) * torch.empty(B, device=dev).uniform_(-1, 1)
    ang = torch.empty(B, device=dev).uniform_(-15, 15) * math.pi / 180
    flip = torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.stack([torch.stack([cos * sw * flip, -sin * sh, tx], 1),
                         torch.stack([sin * sw * flip, cos * sh, ty], 1)], 1)
    grid = F.affine_grid(theta, (B, 3, IMG_SIZE, IMG_SIZE), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    b = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
    c = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
    s = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
    x = x * b
    gray = (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3])
    x = (x - gray.mean((2, 3), keepdim=True)) * c + gray.mean((2, 3), keepdim=True)
    gray = (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3])
    x = gray + (x - gray) * s
    return x.clamp(0, 1)

def eval_view(xb):
    x = xb.float() / 255.0
    return F.interpolate(x, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", antialias=True, align_corners=False)

### 7.1 Fourier low-frequency amplitude mixing (the style-perturbed view)

In [ ]:
def fourier_mix(x, beta=FOURIER_BETA, max_mix=FOURIER_MAX_MIX):
    """Low-frequency amplitude mixing (FDA/FACT-style): swap 'style', keep phase (structure). x in [0,1]."""
    B, C, Hh, Ww = x.shape
    f = torch.fft.fft2(x.float(), norm="ortho")
    amp, pha = f.abs(), f.angle()
    amp = torch.fft.fftshift(amp, dim=(-2, -1))
    h, w = max(1, int(Hh * beta)), max(1, int(Ww * beta))
    ch, cw = Hh // 2, Ww // 2
    perm = torch.randperm(B, device=x.device)
    lam = torch.rand(B, 1, 1, 1, device=x.device) * max_mix
    win = (slice(None), slice(None), slice(ch - h, ch + h), slice(cw - w, cw + w))
    mixed = amp.clone()
    mixed[win] = (1 - lam) * amp[win] + lam * amp[perm][win]
    mixed = torch.fft.ifftshift(mixed, dim=(-2, -1))
    out = torch.fft.ifft2(mixed * torch.exp(1j * pha), norm="ortho").real
    return out.clamp(0, 1)

### 7.2 Loss: cross-entropy + prediction consistency + attention consistency

In [ ]:
def js_div(p, q, eps=1e-8):
    m = 0.5 * (p + q)
    return 0.5 * ((p * ((p + eps) / (m + eps)).log()).sum(-1) + (q * ((q + eps) / (m + eps)).log()).sum(-1))

def compute_loss(model, x01, y, spec):
    ce = lambda lg: F.cross_entropy(lg, y, label_smoothing=0.1)
    if not spec.get("fourier"):
        logits, _ = model(x01)
        return ce(logits.float())
    x2 = fourier_mix(x01)
    logits, maps = model(torch.cat([x01, x2]))
    l1, l2 = logits.float().chunk(2)
    loss = 0.5 * (ce(l1) + ce(l2))
    if spec.get("cons_pred"):
        a, b = F.log_softmax(l1, 1), F.log_softmax(l2, 1)
        loss = loss + LAMBDA_KL * 0.5 * (F.kl_div(a, b, log_target=True, reduction="batchmean") +
                                         F.kl_div(b, a, log_target=True, reduction="batchmean"))
    if spec.get("cons_att") and maps is not None:
        m1, m2 = maps.chunk(2)
        ar = torch.arange(len(y), device=y.device)
        loss = loss + LAMBDA_ATT * js_div(m1[ar, y].flatten(1), m2[ar, y].flatten(1)).mean()
    return loss

### 7.3 Prediction helper

In [ ]:
CACHE_T = torch.from_numpy(CACHE)

def batches(ix, bs):
    for i in range(0, len(ix), bs):
        yield ix[i:i + bs]

@torch.no_grad()
def predict_logits(model, ix, bs=64):
    model.eval(); out = []
    for b in batches(np.asarray(ix), bs):
        x = eval_view(CACHE_T[b].to(DEVICE, non_blocking=True))
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            lg, _ = model(x)
        if not torch.isfinite(lg).all():          # fp16 overflow guard -> redo this batch in fp32
            lg, _ = model(x)
        out.append(lg.float().cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, 0), np.float32)

### 7.4 Training loop

In [ ]:
def train_model(spec, tr, va, classes, seed):
    """Returns (best model, history). `classes` = global class ids, in local order."""
    set_seed(seed)
    K = len(classes)
    g2l = torch.full((len(CLASSES),), -1, dtype=torch.long); g2l[torch.tensor(classes)] = torch.arange(K)
    y_tr = g2l[Y_ALL[tr]]; y_va = g2l[Y_ALL[va]].numpy()
    model = Net(spec, K).to(DEVICE)
    model.mix_on = bool(spec.get("mixstyle"))
    hp = {id(p) for p in model.head_params()}
    base_lr = BACKBONES[spec["backbone"]]["lr"]
    opt = torch.optim.AdamW([
        {"params": [p for p in model.parameters() if id(p) not in hp], "lr": base_lr},
        {"params": [p for p in model.parameters() if id(p) in hp], "lr": base_lr * 10}], weight_decay=0.05)
    steps = max(1, math.ceil(len(tr) / BATCH)); total = EPOCHS * steps
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / steps) *
                                              (0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total)))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    cls_n = Counter(y_tr.tolist()); dom = D_ALL[tr]; dom_n = Counter(dom.tolist())
    w = torch.tensor([cls_n[int(c)] ** -0.5 * dom_n[int(d)] ** -0.5 for c, d in zip(y_tr, dom)], dtype=torch.float)
    tr_t = torch.as_tensor(tr)
    best, best_state, hist = -1.0, None, []
    swa, swa_start = None, int(EPOCHS * SWA_START)
    for ep in range(EPOCHS):
        model.train(); t_ep = time.time(); run_loss = 0.0
        draw = torch.multinomial(w, steps * BATCH, replacement=True)
        for s in range(steps):
            sel = draw[s * BATCH:(s + 1) * BATCH]
            xb = CACHE_T[tr_t[sel]].to(DEVICE, non_blocking=True)
            yb = y_tr[sel].to(DEVICE)
            x = gpu_augment(xb)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                loss = compute_loss(model, x, yb, spec)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item()
        va_pred = predict_logits(model, va).argmax(1)
        f1 = f1_score(y_va, va_pred, average="macro")
        hist.append(dict(epoch=ep, loss=run_loss / steps, val_macro_f1=f1, sec=time.time() - t_ep))
        if f1 > best:
            best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if spec.get("swa") and ep >= swa_start:
            if swa is None:
                swa = torch.optim.swa_utils.AveragedModel(model)
            swa.update_parameters(model)
    if swa is not None:   # SWA: use the weight average (not the best epoch), then recompute BatchNorm statistics
        model = swa.module
        bn_update(model, tr_t, w)
        hist.append(dict(epoch="swa", val_macro_f1=f1_score(y_va, predict_logits(model, va).argmax(1), average="macro")))
        return model, hist
    model.load_state_dict(best_state)
    return model, hist

@torch.no_grad()
def bn_update(model, tr_t, w, n_batches=50):
    """Recompute BatchNorm running statistics for an averaged (SWA) model on augmented training batches."""
    bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
    if not bns:
        return
    for m in bns:
        m.reset_running_stats(); m.momentum = None      # cumulative average
    model.train()
    for _ in range(n_batches):
        sel = torch.multinomial(w, BATCH, replacement=True)
        x = gpu_augment(CACHE_T[tr_t[sel]].to(DEVICE, non_blocking=True))
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            model(x)
    model.eval()

### 7.5 Foundation models: cached frozen features + linear probe

Each encoder runs **once** over all images (the same evaluation view as the other models) and the features are cached in `results/features/`. Per split, a linear classifier is trained on standardized features with the **same class × source balancing** as the other models. Weight decay is chosen on the source validation split only. These baselines need no fine-tuning, so they cost minutes, not hours.

In [ ]:
FM_PARAMS = {}

def load_encoder(spec):
    if spec["kind"] == "timm":
        m = timm.create_model(spec["name"], pretrained=PRETRAINED, num_classes=0, img_size=IMG_SIZE)
        cfg = timm.data.resolve_data_config({}, model=m)
        return m, cfg["mean"], cfg["std"], m
    try:
        import open_clip
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=False)
        import open_clip
    if not PRETRAINED:
        raise RuntimeError("open_clip model needs pretrained weights (Internet)")
    model, _, _ = open_clip.create_model_and_transforms(spec["name"])
    enc = model.visual
    class _Wrap(nn.Module):
        def __init__(s, v): super().__init__(); s.v = v
        def forward(s, x): return s.v(x)
    return _Wrap(enc), open_clip.OPENAI_DATASET_MEAN, open_clip.OPENAI_DATASET_STD, enc

@torch.no_grad()
def fm_features(model_name):
    spec = ALL_MODELS[model_name]
    tag = re.sub(r"[^A-Za-z0-9]", "_", model_name)
    f = FEATS / f"{tag}_{len(idx)}_{hashlib.md5(''.join(idx.rel).encode()).hexdigest()[:10]}.npy"
    pf = f.with_suffix(".params.json")
    if f.exists():
        if pf.exists():
            FM_PARAMS[model_name] = json.loads(pf.read_text())["params_m"]
        return np.load(f).astype(np.float32)
    enc, mean, std, counted = load_encoder(spec)
    enc = enc.to(DEVICE).eval()
    FM_PARAMS[model_name] = count_params(counted)
    pf.write_text(json.dumps({"params_m": FM_PARAMS[model_name]}))
    mean = torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1); std = torch.tensor(std, device=DEVICE).view(1, 3, 1, 1)
    keep = meta.index.values
    out = None
    for b in tqdm(list(batches(keep, 128)), desc=f"features {model_name}"):
        x = (eval_view(CACHE_T[b].to(DEVICE)) - mean) / std
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
            z = enc(x)
        z = z.float()
        if out is None:
            out = np.zeros((len(idx), z.shape[1]), np.float16)
        out[b] = z.cpu().numpy().astype(np.float16)
    del enc; torch.cuda.empty_cache()
    np.save(f, out)
    return out.astype(np.float32)

def train_probe(X, tr, va, te, classes, seed, epochs=300):
    """Class x source balanced linear probe on standardized frozen features. Returns val/test logits + info."""
    set_seed(seed)
    K = len(classes)
    g2l = torch.full((len(CLASSES),), -1, dtype=torch.long); g2l[torch.tensor(classes)] = torch.arange(K)
    Xt = torch.tensor(X[tr], device=DEVICE); mu, sd = Xt.mean(0), Xt.std(0) + 1e-6
    norm = lambda a: (torch.tensor(X[a], device=DEVICE) - mu) / sd
    Xt, Xv, Xe = (Xt - mu) / sd, norm(va), norm(te)
    yt, yv = g2l[Y_ALL[tr]].to(DEVICE), g2l[Y_ALL[va]].numpy()
    cls_n = Counter(yt.tolist()); dom = D_ALL[tr]; dom_n = Counter(dom.tolist())
    w = torch.tensor([cls_n[int(c)] ** -0.5 * dom_n[int(d)] ** -0.5 for c, d in zip(yt.tolist(), dom)],
                     dtype=torch.float, device=DEVICE)
    w = w / w.mean()
    best = (-1, None, None)
    for wd in (1e-4, 1e-3, 1e-2, 1e-1):
        lin = nn.Linear(Xt.shape[1], K).to(DEVICE)
        opt = torch.optim.AdamW(lin.parameters(), lr=1e-3, weight_decay=wd)
        for _ in range(epochs):
            idx_p = torch.randperm(len(Xt), device=DEVICE)
            for s in range(0, len(Xt), 1024):
                bi = idx_p[s:s + 1024]
                loss = (F.cross_entropy(lin(Xt[bi]), yt[bi], reduction="none", label_smoothing=0.1) * w[bi]).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            f1 = f1_score(yv, lin(Xv).argmax(1).cpu().numpy(), average="macro")
        if f1 > best[0]:
            best = (f1, wd, lin)
    lin = best[2]
    with torch.no_grad():
        return lin(Xv).cpu().numpy(), lin(Xe).cpu().numpy(), dict(epoch="probe", weight_decay=best[1], val_macro_f1=best[0])

## 8. Run all experiments (priority order, resumable, time-guarded)

Priority: (1) LODO seed 0 for all main models (foundation-model probes first: minutes) → (2) inner leave-one-source-out runs for cross-source calibration → (3) in-domain CV → (4) ablations seed 0 → (5) LODO other seeds → (6) ablation other seeds.
Every run saves `runs/<key>.json` (config, history, timing) and `runs/<key>.npz` (val/test indices + logits). All analyses below are computed from these files. Seed-0 LODO weights are saved for the XAI section.

In [ ]:
def run_key(model_name, proto, split, seed):
    return re.sub(r"[^A-Za-z0-9_.+-]", "_", f"{model_name}__{proto}__{split}__s{seed}")

def run_done(key):
    """True only if the run exists AND was produced with the current splits/budget (RUN_TAG)."""
    f = RUNS / f"{key}.json"
    if not f.exists():
        return False
    try:
        return json.loads(f.read_text()).get("run_tag") == RUN_TAG
    except Exception:
        return False

### 8.1 Build the run plan (priority order)

In [ ]:
PLAN = []
lodo_splits = [k for k in SPLITS if k[0] == "lodo"]
cv_splits = [k for k in SPLITS if k[0] == "cv"]
inner_splits = [k for k in SPLITS if k[0] == "inner"]
fm_first = [m for m in MAIN_MODELS if MAIN_MODELS[m]["head"] == "probe"]      # minutes -> run first
for m in fm_first + [m for m in MAIN_MODELS if m not in fm_first]:
    for sp in lodo_splits: PLAN.append((m, sp, SEEDS[0]))
if RUN_CROSS_SOURCE_CAL:
    for m in [m for m in CROSS_CAL_MODELS if m in MAIN_MODELS]:
        for sp in inner_splits: PLAN.append((m, sp, SEEDS[0]))
if RUN_CV:
    for m in CV_MODELS:
        for sp in cv_splits: PLAN.append((m, sp, SEEDS[0]))
if RUN_ABLATIONS:
    for m in ABLATION_MODELS:
        for sp in lodo_splits: PLAN.append((m, sp, SEEDS[0]))
for s in SEEDS[1:]:
    for m in MAIN_MODELS:
        for sp in lodo_splits: PLAN.append((m, sp, s))
if RUN_ABLATIONS:
    for s in [s for s in ABLATION_SEEDS if s != SEEDS[0]]:
        for m in ABLATION_MODELS:
            for sp in lodo_splits: PLAN.append((m, sp, s))

done = sum(run_done(run_key(m, *sp, s)) for m, sp, s in PLAN)
stale = sum((RUNS / f"{run_key(m, *sp, s)}.json").exists() and not run_done(run_key(m, *sp, s)) for m, sp, s in PLAN)
if stale:
    print(f"{stale} existing run files were made with different splits/settings (e.g. SMOKE) -> they will be retrained")
print(f"plan: {len(PLAN)} runs | already done: {done} | to do: {len(PLAN) - done}")

### 8.2 Train! (skips finished runs, stops at the time budget)

In [ ]:
SKIPPED = set()
for i, (m, (proto, split), seed) in enumerate(PLAN):
    key = run_key(m, proto, split, seed)
    if run_done(key) or m in SKIPPED:
        continue
    if hours() > TIME_BUDGET_H:
        print(f"Time budget reached ({hours():.1f} h). Stopping; re-run with this output attached to resume.")
        break
    S = SPLITS[(proto, split)]
    spec = ALL_MODELS[m]
    classes = sorted(set(int(c) for c in Y_ALL[S["train"]].tolist()))
    t = time.time()
    if spec["head"] == "probe":
        try:
            Xf = fm_features(m)
        except Exception as e:
            print(f"{m}: encoder unavailable ({e}) -> skipped"); SKIPPED.add(m); continue
        va_logits, te_logits, info = train_probe(Xf, S["train"], S["val"], S["test"], classes, seed)
        hist, model, n_params = [info], None, FM_PARAMS.get(m, float("nan"))
    else:
        model, hist = train_model(spec, S["train"], S["val"], classes, seed)
        va_logits = predict_logits(model, S["val"]); te_logits = predict_logits(model, S["test"])
        n_params = count_params(model)
    np.savez_compressed(RUNS / f"{key}.npz", val_idx=S["val"], val_logits=va_logits,
                        test_idx=S["test"], test_logits=te_logits, classes=np.array(classes))
    if proto == "lodo" and seed == SEEDS[0] and model is not None:
        torch.save(model.state_dict(), WEIGHTS / f"{key}.pt")
    y_te = Y_ALL[S["test_id"]].numpy()
    pos = {g: i for i, g in enumerate(S["test"])}
    pred_g = np.array(classes)[te_logits.argmax(1)]
    pid = pred_g[[pos[g] for g in S["test_id"]]]
    f1 = f1_score(y_te, pid, labels=sorted(set(y_te)), average="macro") if len(y_te) else float("nan")
    save_json(dict(key=key, model=m, spec=spec, protocol=proto, split=split, seed=seed, classes=classes,
                   epochs=EPOCHS, history=hist, minutes=(time.time() - t) / 60, test_id_macro_f1=f1,
                   params_m=n_params, run_tag=RUN_TAG), RUNS / f"{key}.json")
    print(f"[{i + 1}/{len(PLAN)}] {key}: macro-F1(ID test)={f1:.3f} | {(time.time() - t) / 60:.1f} min | elapsed {hours():.2f} h")
    del model; torch.cuda.empty_cache()

remaining = [p for p in PLAN if p[0] not in SKIPPED and not run_done(run_key(p[0], *p[1], p[2]))]
if SKIPPED:
    print("skipped (encoder unavailable):", sorted(SKIPPED))
print("ALL RUNS DONE" if not remaining else f"{len(remaining)} runs remaining -> attach this output and run again")

## 9. Generalization results (RQ1, RQ2) and statistics

- **Macro-F1 / accuracy** on each held-out dataset's ID images (classes seen in training); mean ± std over seeds.
- **In-domain** macro-F1 from CV on the same dataset and the same classes → **degradation** (RQ1).
- **Proposed vs each baseline:** Wilcoxon signed-rank over (held-out × seed) macro-F1 pairs; exact **McNemar** per held-out set (seed 0); Holm correction; group-bootstrap 95% CI for macro-F1.

In [ ]:
def load_run(key):
    j = json.loads((RUNS / f"{key}.json").read_text())
    z = np.load(RUNS / f"{key}.npz")
    return j, {k: z[k] for k in z.files}

def softmax(z, T=1.0):
    z = z / T; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

### 9.1 Load all finished runs

In [ ]:
RUN_INDEX = []
for f in sorted(RUNS.glob("*.json")):
    j = json.loads(f.read_text())
    if j["model"] in ALL_MODELS and j.get("run_tag") == RUN_TAG:
        RUN_INDEX.append(j)
RI = pd.DataFrame(RUN_INDEX)
print("runs available:", len(RI))

### 9.2 Evaluate every LODO run on held-out ID classes

In [ ]:
def lodo_eval(j, z):
    """Metrics on the ID part of a LODO test set (+ per-sample correctness for paired tests)."""
    S = SPLITS[("lodo", j["split"])]
    classes = np.array(z["classes"])
    pos = {g: i for i, g in enumerate(z["test_idx"])}
    sel = np.array([pos[g] for g in S["test_id"]], dtype=int)
    y = Y_ALL[S["test_id"]].numpy()
    pred = classes[z["test_logits"][sel].argmax(1)]
    labels = sorted(set(y))
    return dict(macro_f1=f1_score(y, pred, labels=labels, average="macro"),
                acc=accuracy_score(y, pred),
                per_class=dict(zip([CLASSES[c] for c in labels], f1_score(y, pred, labels=labels, average=None))),
                correct=(pred == y), test_id=S["test_id"], y=y, pred=pred)

LODO = {}
for j in RUN_INDEX:
    if j["protocol"] == "lodo":
        jj, z = load_run(j["key"])
        LODO[(j["model"], j["split"], j["seed"])] = lodo_eval(jj, z)

def lodo_table(models):
    rows = []
    for m in models:
        for held in dss:
            vals = [LODO[k] for k in LODO if k[0] == m and k[1] == held]
            if not vals:
                continue
            f1s, accs = [v["macro_f1"] for v in vals], [v["acc"] for v in vals]
            rows.append(dict(model=m, held_out=held, n_seeds=len(vals),
                             macro_f1=np.mean(f1s), macro_f1_std=np.std(f1s), acc=np.mean(accs)))
    return pd.DataFrame(rows)

### 9.3 Main table: LODO macro-F1

In [ ]:
lt = lodo_table(MAIN_MODELS)
if len(lt):
    piv = lt.pivot(index="model", columns="held_out", values="macro_f1")
    piv["mean"] = piv.mean(axis=1)
    pivs = lt.pivot(index="model", columns="held_out", values="macro_f1_std")
    cell = piv.drop(columns="mean").round(3).astype(str) + " ± " + pivs.round(3).astype(str)
    cell["mean"] = piv["mean"].round(3)
    cell.to_csv(TABLES / "lodo_macro_f1.csv")
    report("LODO macro-F1 on held-out ID classes (mean ± std over seeds)", md_table(cell))

### 9.4 RQ1: in-domain vs cross-source -> degradation

In [ ]:
# ---------- in-domain CV (pooled over folds) on the LODO class subset -> degradation
cv_rows = []
for m in MAIN_MODELS:
    keys = [j["key"] for j in RUN_INDEX if j["model"] == m and j["protocol"] == "cv"]
    if len(keys) < CV_FOLDS:
        continue
    gi, gp = [], []
    for k in keys:
        j, z = load_run(k)
        gi.append(z["test_idx"]); gp.append(np.array(z["classes"])[z["test_logits"].argmax(1)])
    gi, gp = np.concatenate(gi), np.concatenate(gp)
    for held in dss:
        shared_cls = set(Y_ALL[SPLITS[("lodo", held)]["test_id"]].tolist())
        msk = (idx.dataset.values[gi] == held) & np.isin(Y_ALL.numpy()[gi], list(shared_cls))
        if msk.sum() == 0:
            continue
        y = Y_ALL.numpy()[gi][msk]
        cv_rows.append(dict(model=m, held_out=held,
                            in_domain_f1=f1_score(y, gp[msk], labels=sorted(shared_cls), average="macro")))
cvt = pd.DataFrame(cv_rows)
if len(cvt) and len(lt):
    deg = cvt.merge(lt[["model", "held_out", "macro_f1"]], on=["model", "held_out"])
    deg["drop_points"] = 100 * (deg.in_domain_f1 - deg.macro_f1)
    deg.to_csv(TABLES / "degradation.csv", index=False)
    dp = deg.pivot(index="model", columns="held_out", values="drop_points"); dp["mean"] = dp.mean(axis=1)
    report("RQ1 — cross-source degradation (in-domain - LODO macro-F1, percentage points)", md_table(dp, "{:.1f}"))
    big = dp["mean"].mean()
    report("RQ1 checkpoint", f"Mean drop across models = **{big:.1f} points** "
           + ("(large: cross-source generalization is a real problem)" if big >= 15 else
              "(< 15: shift the paper's emphasis toward uncertainty gating and remedies)"))

### 9.5 RQ2: ablation

In [ ]:
# ---------- ablation table
at = lodo_table([m for m in ABL_CHAIN if m in ALL_MODELS])
if len(at):
    ap = at.pivot(index="model", columns="held_out", values="macro_f1").reindex([m for m in ABL_CHAIN if m in set(at.model)])
    ap["mean"] = ap.mean(axis=1)
    ap.to_csv(TABLES / "ablation_macro_f1.csv")
    report("RQ2 — ablation (LODO macro-F1, mean over available seeds)", md_table(ap))

### 9.6 Statistical tests: SILA-Net vs each baseline

In [ ]:
# ---------- statistics: proposed vs baselines
def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p)
    run = 0
    for r, i in enumerate(o):
        run = max(run, min(1, (len(p) - r) * p[i])); adj[i] = run
    return adj

def group_bootstrap_f1(ev, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    groups = idx.group.values[ev["test_id"]]
    ug = np.unique(groups); gi = {g: np.where(groups == g)[0] for g in ug}
    labels = sorted(set(ev["y"])); out = []
    for _ in range(n):
        s = np.concatenate([gi[g] for g in rng.choice(ug, len(ug), replace=True)])
        out.append(f1_score(ev["y"][s], ev["pred"][s], labels=labels, average="macro"))
    return np.percentile(out, [2.5, 97.5])

stat_rows = []
for b in [m for m in MAIN_MODELS if m != PROPOSED]:
    pairs = [(LODO[(PROPOSED, h, s)]["macro_f1"], LODO[(b, h, s)]["macro_f1"])
             for h in dss for s in SEEDS if (PROPOSED, h, s) in LODO and (b, h, s) in LODO]
    if len(pairs) < 2:
        continue
    a, c = np.array(pairs).T
    try:
        p_w = stats.wilcoxon(a, c).pvalue if np.any(a != c) else 1.0
    except ValueError:
        p_w = float("nan")
    for h in dss:
        k1, k2 = (PROPOSED, h, SEEDS[0]), (b, h, SEEDS[0])
        if k1 in LODO and k2 in LODO:
            c1, c2 = LODO[k1]["correct"], LODO[k2]["correct"]
            n01, n10 = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
            p_m = stats.binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
            stat_rows.append(dict(baseline=b, held_out=h, proposed_only_correct=n01, baseline_only_correct=n10,
                                  mcnemar_p=p_m, mean_diff_f1=float(np.mean(a - c)), wilcoxon_p=p_w, n_pairs=len(a)))
st = pd.DataFrame(stat_rows)
if len(st):
    st["mcnemar_p_holm"] = holm(st.mcnemar_p.values)
    st.to_csv(TABLES / "stats_proposed_vs_baselines.csv", index=False)
    report(f"Statistics — {PROPOSED} vs baselines", md_table(st, "{:.4f}"))

### 9.7 Bootstrap confidence intervals and per-class F1

In [ ]:
ci_rows = []
for m in MAIN_MODELS:
    for h in dss:
        k = (m, h, SEEDS[0])
        if k in LODO:
            lo, hi = group_bootstrap_f1(LODO[k], n=200 if SMOKE else 1000)
            ci_rows.append(dict(model=m, held_out=h, macro_f1=LODO[k]["macro_f1"], ci_low=lo, ci_high=hi))
if ci_rows:
    pd.DataFrame(ci_rows).to_csv(TABLES / "bootstrap_ci_seed0.csv", index=False)

pc_rows = [dict(model=m, held_out=h, seed=s, **{f"F1_{c}": v for c, v in ev["per_class"].items()})
           for (m, h, s), ev in LODO.items() if m in MAIN_MODELS]
if pc_rows:
    pc = pd.DataFrame(pc_rows).groupby(["model", "held_out"]).mean(numeric_only=True).drop(columns="seed")
    pc.to_csv(TABLES / "per_class_f1_lodo.csv")
    report("Per-class F1 on held-out sets (mean over seeds)", md_table(pc))

### 9.8 Confusion matrices (SILA-Net, seed 0)

In [ ]:
# confusion matrices for the proposed model (seed 0)
avail = [h for h in dss if (PROPOSED, h, SEEDS[0]) in LODO]
if avail:
    fig, axes = plt.subplots(1, len(avail), figsize=(4.6 * len(avail), 4.2), squeeze=False)
    for ax, h in zip(axes[0], avail):
        ev = LODO[(PROPOSED, h, SEEDS[0])]
        labs = sorted(set(ev["y"]) | set(ev["pred"]))
        cm = confusion_matrix(ev["y"], ev["pred"], labels=labs)
        ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(labs))); ax.set_xticklabels([CLASSES[l] for l in labs], rotation=60, ha="right", fontsize=7)
        ax.set_yticks(range(len(labs))); ax.set_yticklabels([CLASSES[l] for l in labs], fontsize=7)
        for (r, c), v in np.ndenumerate(cm):
            if v: ax.text(c, r, v, ha="center", va="center", fontsize=6)
        ax.set_title(f"held-out: {h}", fontsize=9); ax.set_xlabel("predicted"); ax.set_ylabel("true")
    plt.tight_layout(); plt.savefig(FIGS / "confusion_proposed_lodo.png", dpi=120); plt.show()

## 10. Uncertainty: calibration, conformal prediction and gating (RQ3)

The farmer acts on a **treatment**, not on a disease name. Different diseases often share one management action (purple blotch, Stemphylium and Botrytis → foliar fungicide programme), so confusing them is harmless, while confusing a viral disease (thrips control) with a fungal one is not. Our RQ3 design follows from this.

**Calibration sources** (where the conformal threshold comes from; the held-out region is never used):
| name | threshold computed on | meaning |
|---|---|---|
| `val` | pooled source validation split | standard split conformal; assumes test looks like training (it does not) |
| `worst` | per-source validation quantile, maximum | guards the hardest *known* source |
| `xsrc` (leave-one-source-out) | the main model's own validation scores, at a **stricter level α′** learned from inner models (section 5.1b): α′ is the largest level whose validation threshold still covers ≥ 1-α of a training source the inner model **never saw** (worst source) | calibrates for *new-region* shift. Only the *level* is transferred, never the raw threshold, so differences in probability scale between models do not matter |

**Gates** (a remedy is issued only if the gate accepts; otherwise "re-take photo / ask the extension officer"):
| name | prediction set | issue a remedy when |
|---|---|---|
| `disease` | disease set, LAC score `1 - p_y` | exactly one disease |
| `remedy-agg` | same disease set | all diseases in the set share **one** remedy group |
| `remedy-CP` (CP over treatment groups; cf. conformal structured prediction) | conformal set over **remedy groups**: `P(g) = Σ p(k) over k in g`, score `1 - P(g_y)` | exactly one remedy group |

For `remedy-agg` and `remedy-CP`, the conformal guarantee holds for the **remedy** (true remedy in the set with probability ≥ 1-α under exchangeability). Gating on remedies instead of diseases therefore keeps safety while withholding less often.

**Reported:** acceptance rate, remedy-set coverage, **wrong remedies issued** (issued group ≠ true group, including images of diseases the model never saw), wrong-issue rate, share of unseen-disease images withheld; plus temperature scaling ECE, AURC and a max-softmax gate as a non-conformal reference.

In [ ]:
REMEDY_GROUP = {   # diseases that share the same management action (see the knowledge base in section 12)
    "healthy": "none", "purple_blotch": "fungal_foliar", "stemphylium": "fungal_foliar",
    "botrytis": "fungal_foliar", "downy_mildew": "oomycete", "rust": "rust",
    "viral": "vector_control", "iris_yellow_virus": "vector_control", "virosis": "vector_control",
    "xanthomonas": "bacterial", "fusarium": "soilborne", "bulb_blight": "soilborne", "burn_tip": "abiotic_tip",
}

### 10.1 Calibration and conformal helpers

In [ ]:
def fit_temperature(logits, y):
    """Temperature scaling on the source validation split; T is bounded to [0.05, 20] and the fit is
    rejected (T = 1) if it does not lower validation NLL (guards against LBFGS divergence on small sets)."""
    lg = torch.tensor(logits, dtype=torch.float64); yt = torch.tensor(y)
    nll = lambda T: float(F.cross_entropy(lg / T, yt))
    t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([t], lr=0.05, max_iter=100, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(lg / t.clamp(-3, 3).exp(), yt); loss.backward(); return loss
    try:
        opt.step(closure)
        T = float(t.detach().clamp(-3, 3).exp())
    except Exception:
        T = 1.0
    if not np.isfinite(T) or nll(T) > nll(1.0):
        T = 1.0
    return T

def ece(p, y, bins=15):
    conf, pred = p.max(1), p.argmax(1); e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e

def conf_quantile(scores, alpha):
    n = len(scores)
    if n == 0:
        return 1.0
    q = min(1.0, math.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(scores, q, method="higher"))

def aurc(conf, correct):
    o = np.argsort(-conf); err = 1 - correct[o].astype(float)
    return float(np.mean(np.cumsum(err) / np.arange(1, len(err) + 1)))

GROUPS = sorted(set(REMEDY_GROUP.values()))
G2I = {g: i for i, g in enumerate(GROUPS)}
CLASS_GROUP = np.array([G2I[REMEDY_GROUP[c]] for c in CLASSES])     # global class id -> remedy group id

def group_probs(p, classes):
    """Disease probabilities (over the run's classes) -> remedy-group probabilities."""
    M = np.zeros((len(classes), len(GROUPS)))
    M[np.arange(len(classes)), CLASS_GROUP[classes]] = 1
    return p @ M

def run_probs(key):
    """Load a run, fit temperature on its source validation split, return calibrated val/test probabilities."""
    j, z = load_run(key)
    classes = np.array(z["classes"]); g2l = {g: i for i, g in enumerate(classes)}
    yv = np.array([g2l[g] for g in Y_ALL[z["val_idx"]].tolist()])
    T = fit_temperature(z["val_logits"], yv)
    return j, z, classes, T, softmax(z["val_logits"], T), yv

def scores(p, classes, y_global):
    """Nonconformity scores: disease (1 - p_y) and remedy (1 - P(g_y)). y_global must be in `classes`."""
    g2l = {g: i for i, g in enumerate(classes)}
    yl = np.array([g2l[g] for g in y_global]); ar = np.arange(len(yl))
    return 1 - p[ar, yl], 1 - group_probs(p, classes)[ar, CLASS_GROUP[y_global]]

XSRC = {}
ALPHA_GRID = np.concatenate([[0.0], np.geomspace(1e-3, ALPHA, 60)])

def cross_source_alpha(model_name, held):
    """Cross-source LEVEL calibration. For every inner model (trained without source S), find the largest
    nominal level a' whose threshold - computed on the inner model's OWN source-validation scores - still
    covers >= 1-ALPHA of the unseen source S. Take the worst (smallest) a' over the training sources.
    The main model then uses its own validation quantile at a'. Transferring the level instead of the raw
    threshold makes the procedure independent of each model's probability scale / number of classes.
    Returns (a'_disease, a'_remedy, per-source details) or None if inner runs are missing."""
    if (model_name, held) in XSRC:
        return XSRC[(model_name, held)]
    best_d, best_r, detail = [], [], []
    for src in [d for d in dss if d != held]:
        k = run_key(model_name, "inner", f"{held}|{src}", SEEDS[0])
        if not run_done(k):
            XSRC[(model_name, held)] = None
            return None
        _, z, classes, T, pv, yv = run_probs(k)
        S = SPLITS[("inner", f"{held}|{src}")]
        pos = {g: i for i, g in enumerate(z["test_idx"])}
        sel = np.array([pos[g] for g in S["test_id"]], dtype=int)
        if len(sel) == 0:
            continue
        vd, vr = scores(pv, classes, classes[yv])
        td, tr_ = scores(softmax(z["test_logits"][sel], T), classes, Y_ALL[S["test_id"]].numpy())
        pick = []
        for v, t in ((vd, td), (vr, tr_)):
            ok = [a for a in ALPHA_GRID if (t <= conf_quantile(v, a)).mean() >= 1 - ALPHA]
            pick.append(max(ok) if ok else 0.0)
        best_d.append(pick[0]); best_r.append(pick[1])
        detail.append(dict(source=src, alpha_disease=pick[0], alpha_remedy=pick[1],
                           naive_cov_disease=(td <= conf_quantile(vd, ALPHA)).mean(),
                           naive_cov_remedy=(tr_ <= conf_quantile(vr, ALPHA)).mean()))
    XSRC[(model_name, held)] = (min(best_d), min(best_r), detail) if best_d else None
    return XSRC[(model_name, held)]

### 10.2 Calibrate, build prediction sets and gate every LODO run

In [ ]:
UQ_ROWS, GATE_ROWS, QHAT = [], [], {}
for j in RUN_INDEX:
    if j["protocol"] != "lodo":
        continue
    m, held, seed = j["model"], j["split"], j["seed"]
    _, z, classes, T, pv, yv = run_probs(j["key"])
    S = SPLITS[("lodo", held)]
    sv_d, sv_r = scores(pv, classes, classes[yv])
    dv = idx.dataset.values[z["val_idx"]]
    doms = [d for d in np.unique(dv) if (dv == d).sum() >= 20]
    Q = {"val": (conf_quantile(sv_d, ALPHA), conf_quantile(sv_r, ALPHA)),
         "worst": (max([conf_quantile(sv_d[dv == d], ALPHA) for d in doms] or [conf_quantile(sv_d, ALPHA)]),
                   max([conf_quantile(sv_r[dv == d], ALPHA) for d in doms] or [conf_quantile(sv_r, ALPHA)]))}
    xs = cross_source_alpha(m, held)
    if xs is not None:
        Q["xsrc"] = (conf_quantile(sv_d, xs[0]), conf_quantile(sv_r, xs[1]))
    QHAT[(m, held, seed)] = dict(T=T, alpha_xsrc=(xs[0], xs[1]) if xs is not None else None, **Q)

    pt_raw = softmax(z["test_logits"]); pt = softmax(z["test_logits"], T); Pt = group_probs(pt, classes)
    pos = {g: i for i, g in enumerate(z["test_idx"])}
    id_sel = np.array([pos[g] for g in S["test_id"]], dtype=int)
    ood_sel = np.array([pos[g] for g in S["test_ood"]], dtype=int)
    y_id_l = np.searchsorted(classes, Y_ALL[S["test_id"]].numpy())
    true_g = CLASS_GROUP[Y_ALL[z["test_idx"]].numpy()]
    top_g = CLASS_GROUP[classes[pt.argmax(1)]]

    # MSP reference gate: smallest tau whose selective accuracy on val >= 1 - alpha
    cv_, okv = pv.max(1), pv.argmax(1) == yv
    tau = next((t_ for t_ in np.sort(np.unique(cv_)) if (cv_ >= t_).sum() >= 10 and okv[cv_ >= t_].mean() >= 1 - ALPHA), 1.0)
    acc_msp = pt.max(1) >= tau
    UQ_ROWS.append(dict(model=m, held_out=held, seed=seed, T=T,
                        ece_raw=ece(pt_raw[id_sel], y_id_l), ece_ts=ece(pt[id_sel], y_id_l),
                        aurc_msp=aurc(pt[id_sel].max(1), pt[id_sel].argmax(1) == y_id_l),
                        no_gate_wrong=int((top_g != true_g).sum()),
                        msp_wrong=int(((top_g != true_g) & acc_msp).sum()), msp_accept_id=acc_msp[id_sel].mean(),
                        n_test=len(pt), n_unseen_class=len(ood_sel)))

    for cal, (qd, qr) in Q.items():
        dset = (1 - pt) <= qd                                   # disease prediction sets
        dset_groups = [set(CLASS_GROUP[classes[np.where(r)[0]]]) for r in dset]
        rset = (1 - Pt) <= qr                                   # remedy-group prediction sets
        gates = {
            "disease":    (dset.sum(1) == 1, top_g, np.array([tg in gs for tg, gs in zip(true_g, dset_groups)])),
            "remedy-agg": (np.array([len(gs) == 1 for gs in dset_groups]),
                           np.array([next(iter(gs)) if len(gs) == 1 else -1 for gs in dset_groups]),
                           np.array([tg in gs for tg, gs in zip(true_g, dset_groups)])),
            "remedy-CP":  (rset.sum(1) == 1, rset.argmax(1), rset[np.arange(len(true_g)), true_g]),
        }
        for gate, (accept, issued, rem_cov) in gates.items():
            wrong = accept & (issued != true_g)
            GATE_ROWS.append(dict(model=m, held_out=held, seed=seed, calibration=cal, gate=gate,
                                  disease_cov_id=dset[id_sel, y_id_l].mean() if len(id_sel) else np.nan,
                                  remedy_cov_id=rem_cov[id_sel].mean() if len(id_sel) else np.nan,
                                  accept_id=accept[id_sel].mean() if len(id_sel) else np.nan,
                                  withheld_unseen=(~accept[ood_sel]).mean() if len(ood_sel) else np.nan,
                                  issued=int(accept.sum()), wrong_issued=int(wrong.sum()),
                                  wrong_rate=wrong.sum() / max(1, accept.sum())))

### 10.3 RQ3 results: coverage, set size, wrong remedies avoided, risk–coverage

In [ ]:
uq, gq = pd.DataFrame(UQ_ROWS), pd.DataFrame(GATE_ROWS)
if len(uq):
    uq.to_csv(TABLES / "uncertainty_calibration_all_runs.csv", index=False)
    gq.to_csv(TABLES / "uncertainty_gates_all_runs.csv", index=False)
    calib = uq[uq.model.isin(MAIN_MODELS)].groupby("model")[["ece_raw", "ece_ts", "aurc_msp"]].mean()
    dcov = (gq[gq.gate == "disease"].groupby(["model", "calibration"]).disease_cov_id.mean().unstack())
    calib = calib.join(dcov.add_prefix("disease_cov_"))
    report(f"RQ3 - calibration and disease-set coverage on held-out regions (target {1 - ALPHA:.0%}; mean over held-out sets and seeds)",
           md_table(calib))

    xs_rows = [dict(model=m, held_out=h, **d) for (m, h), v in XSRC.items() if v is not None for d in v[2]]
    if xs_rows:
        xs_df = pd.DataFrame(xs_rows); xs_df.to_csv(TABLES / "rq3_cross_source_level_calibration.csv", index=False)
        report(f"RQ3 - standard conformal on a never-seen source (inner models): nominal {1 - ALPHA:.0%} coverage "
               "actually achieved, and the stricter level a' needed to restore it",
               md_table(xs_df.groupby("model")[["naive_cov_disease", "naive_cov_remedy",
                                                "alpha_disease", "alpha_remedy"]].mean(), "{:.3f}"))

    # best foundation model by mean LODO F1 (for the comparison set)
    fm_avail = [m for m in FOUNDATION_MODELS if m in set(lt.model)] if len(lt) else []
    best_fm = max(fm_avail, key=lambda m: lt[lt.model == m].macro_f1.mean()) if fm_avail else None
    focus = [m for m in [PROPOSED, "EfficientNetB0", "ConvNeXtT", best_fm] if m and m in set(gq.model)]
    per_seed = (gq[gq.model.isin(focus)].groupby(["model", "calibration", "gate", "seed"])
                .agg(accept_id=("accept_id", "mean"), remedy_cov_id=("remedy_cov_id", "mean"),
                     withheld_unseen=("withheld_unseen", "mean"), issued=("issued", "sum"),
                     wrong_issued=("wrong_issued", "sum")))
    per_seed["wrong_rate"] = per_seed.wrong_issued / per_seed.issued.clip(lower=1)
    gate_tab = per_seed.groupby(["model", "calibration", "gate"]).mean()
    ng = uq[uq.model.isin(focus)].groupby(["model", "seed"]).agg(no_gate_wrong=("no_gate_wrong", "sum"),
                                                                 n=("n_test", "sum")).groupby("model").mean()
    gate_tab = gate_tab.join(ng)
    gate_tab.to_csv(TABLES / "rq3_gate_comparison.csv")
    report("RQ3 - decision-level gating: acceptance vs wrong remedies (summed over held-out sets, mean over seeds)",
           md_table(gate_tab, "{:.3f}"))

    if PROPOSED in set(gq.model):
        g = gate_tab.loc[PROPOSED]
        base = g.loc[("val", "disease")] if ("val", "disease") in g.index else None
        best_key = ("xsrc", "remedy-CP") if ("xsrc", "remedy-CP") in g.index else ("worst", "remedy-CP")
        ours = g.loc[best_key]
        lines = [f"- no gate: **{ours.no_gate_wrong:.0f}** wrong remedies would be issued"]
        if base is not None:
            lines.append(f"- standard conformal (`val`, disease singleton): {base.wrong_issued:.0f} wrong, "
                         f"accepts {base.accept_id:.0%} of held-out images, remedy coverage {base.remedy_cov_id:.0%}")
        lines.append(f"- **ours (`{best_key[0]}`, remedy-CP)**: {ours.wrong_issued:.0f} wrong, accepts "
                     f"{ours.accept_id:.0%}, remedy coverage {ours.remedy_cov_id:.0%}, withholds "
                     f"{ours.withheld_unseen:.0%} of unseen-disease images")
        report(f"RQ3 headline ({PROPOSED})", "\n".join(lines))

    # trade-off figure: acceptance (x) vs wrong-issue rate (y) for every calibration x gate
    fig, axes = plt.subplots(1, len(focus), figsize=(4.4 * len(focus), 3.8), squeeze=False)
    mk = {"disease": "o", "remedy-agg": "s", "remedy-CP": "*"}
    col = {"val": "tab:gray", "worst": "tab:blue", "xsrc": "tab:red"}
    for ax, m in zip(axes[0], focus):
        for (cal, gate), r in gate_tab.loc[m].iterrows():
            ax.scatter(r.accept_id, r.wrong_rate, marker=mk[gate], s=120 if gate == "remedy-CP" else 60,
                       color=col[cal], label=f"{cal} / {gate}")
        ax.set_title(m, fontsize=10); ax.set_xlabel("acceptance on held-out ID images"); ax.set_ylabel("wrong-remedy rate")
    axes[0, -1].legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    plt.tight_layout(); plt.savefig(FIGS / "rq3_acceptance_vs_wrong_remedy.png", dpi=120); plt.show()

    # risk-coverage curves, seed 0
    fig, ax = plt.subplots(1, len(dss), figsize=(4.2 * len(dss), 3.4), squeeze=False)
    for a, h in zip(ax[0], dss):
        for m in focus:
            k = run_key(m, "lodo", h, SEEDS[0])
            if not run_done(k):
                continue
            jj, z = load_run(k); S = SPLITS[("lodo", h)]
            pos = {g: i for i, g in enumerate(z["test_idx"])}
            sel = np.array([pos[g] for g in S["test_id"]], dtype=int)
            classes = np.array(z["classes"])
            p = softmax(z["test_logits"][sel]); ok = classes[p.argmax(1)] == Y_ALL[S["test_id"]].numpy()
            o = np.argsort(-p.max(1)); cov = np.arange(1, len(o) + 1) / len(o)
            a.plot(cov, np.cumsum(1 - ok[o]) / np.arange(1, len(o) + 1), label=m)
        a.set_title(f"held-out: {h}", fontsize=9); a.set_xlabel("coverage"); a.set_ylabel("risk")
    ax[0, 0].legend(fontsize=7); plt.tight_layout(); plt.savefig(FIGS / "risk_coverage.png", dpi=120); plt.show()

## 11. Quantitative explainability (RQ4)

For each seed-0 LODO model and each held-out set, on `XAI_SAMPLES` held-out ID images (the same images for all models):
- **Saliency:** Grad-CAM on the last feature map (token Grad-CAM for ViT); for SILA-Net also its own **lesion attention map** of the predicted class.
- **Faithfulness:** deletion AUC (lower is better) and insertion AUC (higher is better), 20 steps, blurred-image baseline; a **random-saliency** reference.
- **Concentration:** normalized saliency entropy; **border mass** (share of saliency in the outer 15% frame; a proxy for looking at background/context instead of the leaf/lesion; cf. the border-activation ratio of Xiang et al. 2026).
- **Failure analysis:** the same metrics for *correct vs wrong* predictions (Mann–Whitney U) → do cross-source errors come with diffuse/off-lesion attention?
- **Lesion alignment:** if a dataset with masks named `<uid>.png` inside a folder `lesion_masks` is attached, energy-based pointing game and IoU of the top-15% saliency are computed (and inter-annotator IoU if `lesion_masks_b` exists). An **annotation pack** of the same images is exported for the team to draw these masks (SAM-assisted).

### 11.1 Grad-CAM (CNNs and ViT)

In [ ]:
from torchvision.transforms.functional import gaussian_blur

def gradcam(model, x01, target):
    """Grad-CAM on the last feature map. ViT: on the input of the last block's attention (norm1 output),
    because the final tokens only reach the classifier through the CLS token (patch-token gradients = 0)."""
    model.eval()
    store, hook = {}, None
    is_vit = BACKBONES[model.spec["backbone"]]["timm"].startswith("vit")
    if is_vit:
        hook = model.backbone.blocks[-1].norm1.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    with torch.enable_grad():
        f = model.features(x01)
        act = store["a"] if is_vit else f
        act.retain_grad()
        logits, _ = model.head(f)
        logits[torch.arange(len(target)), target].sum().backward()
    if hook is not None:
        hook.remove()
    model.zero_grad(set_to_none=True)
    g, f = act.grad.detach(), act.detach()
    if f.dim() == 4:
        cam = F.relu((g.mean((2, 3), keepdim=True) * f).sum(1))
    else:   # ViT tokens
        npre = getattr(model.backbone, "num_prefix_tokens", 1)
        ft, gt = f[:, npre:], g[:, npre:]
        cam = F.relu((gt.mean(1, keepdim=True) * ft).sum(-1))
        s = int(math.sqrt(cam.size(1))); cam = cam.view(-1, s, s)
    cam = F.interpolate(cam.unsqueeze(1), size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False).squeeze(1)
    cam = cam + 1e-6 * cam.flatten(1).max(1).values.clamp(min=1e-6).view(-1, 1, 1)   # all-zero map -> uniform
    return cam / (cam.flatten(1).max(1).values.view(-1, 1, 1) + 1e-8)

### 11.2 SILA-Net lesion-attention maps

In [ ]:
@torch.no_grad()
def attn_map(model, x01, target):
    _, maps = model(x01)
    m = maps[torch.arange(len(target)), target].unsqueeze(1)
    m = F.interpolate(m, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False).squeeze(1)
    return m / (m.flatten(1).max(1).values.view(-1, 1, 1) + 1e-8)

### 11.3 Faithfulness (deletion / insertion) and concentration metrics

In [ ]:
@torch.no_grad()
def del_ins(model, x01, sal, target, steps=20):
    """Deletion / insertion AUC for one image. x01: 3xHxW, sal: HxW."""
    n = sal.numel()
    rank = torch.empty(n, device=sal.device, dtype=torch.long)
    rank[sal.flatten().argsort(descending=True)] = torch.arange(n, device=sal.device)
    ks = torch.linspace(0, n, steps + 1, device=sal.device)
    masks = (rank[None] < ks[:, None]).float().view(steps + 1, 1, IMG_SIZE, IMG_SIZE)
    blur = gaussian_blur(x01.unsqueeze(0), kernel_size=31, sigma=10.0)
    xd = x01 * (1 - masks) + blur * masks
    xi = blur * (1 - masks) + x01 * masks
    pd_ = model(xd)[0].float().softmax(1)[:, target]      # fp32 on purpose (stable probabilities)
    pi_ = model(xi)[0].float().softmax(1)[:, target]
    auc = lambda v: float(((v[1:] + v[:-1]) / 2).mean())
    return auc(pd_), auc(pi_)

def sal_stats(sal):
    p = sal.flatten(1) / (sal.flatten(1).sum(1, keepdim=True) + 1e-8)
    ent = -(p * (p + 1e-12).log()).sum(1) / math.log(p.size(1))
    b = int(0.15 * IMG_SIZE)
    inner = sal[:, b:-b, b:-b].flatten(1).sum(1)
    border = 1 - inner / (sal.flatten(1).sum(1) + 1e-8)
    return ent.cpu().numpy(), border.cpu().numpy()

### 11.4 Lesion masks (optional) and the fixed XAI sample

In [ ]:
def find_mask_dir(name):
    if not INPUT_ROOT.exists():
        return None
    for p in INPUT_ROOT.rglob(name):
        if p.is_dir():
            return p
    return None
MASKS_A, MASKS_B = find_mask_dir("lesion_masks"), find_mask_dir("lesion_masks_b")
def uid_of(i):
    return hashlib.md5(idx.rel[i].encode()).hexdigest()[:12]

# fixed XAI / annotation sample per held-out set (stratified by class)
XAI_IDX = {}
for h in dss:
    pool = meta.loc[SPLITS[("lodo", h)]["test_id"]]
    if len(pool) == 0:
        continue
    per = max(1, XAI_SAMPLES // pool.label_name.nunique())
    pick = pool.sample(frac=1.0, random_state=0).groupby("label_name").head(per)
    XAI_IDX[h] = pick.index.values[:XAI_SAMPLES]

### 11.5 Export the annotation pack for the team

In [ ]:
# annotation pack (original files) for the team's lesion masks
pack = OUT / "annotation_pack"
(pack / "images").mkdir(parents=True, exist_ok=True)
pack_rows = []
for h, ix in XAI_IDX.items():
    for i in ix:
        u = uid_of(i); src = Path(idx.path[i])
        shutil.copy2(src, pack / "images" / f"{u}{src.suffix.lower()}")
        pack_rows.append(dict(uid=u, file=f"{u}{src.suffix.lower()}", dataset=idx.dataset[i],
                              label=idx.label_name[i], rel=idx.rel[i]))
pd.DataFrame(pack_rows).to_csv(pack / "annotation_index.csv", index=False)
(pack / "README.txt").write_text(
    "Draw a binary lesion mask for each image (white = lesion, black = rest), same size as the image.\n"
    "Save as <uid>.png in a folder named lesion_masks (annotator A) and lesion_masks_b (annotator B).\n"
    "Upload both folders as a Kaggle dataset and attach it; section 11 then reports pointing game / IoU\n"
    "and inter-annotator IoU. SAM (segment-anything) clicks are fine; correct its output by hand.\n")

### 11.6 Run the XAI evaluation

In [ ]:
def load_mask(mdir, i):
    for ext in (".png", ".jpg"):
        f = mdir / f"{uid_of(i)}{ext}"
        if f.exists():
            m = Image.open(f).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
            return torch.from_numpy(np.asarray(m) > 127)
    return None

XAI_ROWS = []
for m in MAIN_MODELS:
    for h, ix in XAI_IDX.items():
        key = run_key(m, "lodo", h, SEEDS[0])
        wf = WEIGHTS / f"{key}.pt"
        if not wf.exists() or not run_done(key):
            continue
        j = json.loads((RUNS / f"{key}.json").read_text())
        classes = np.array(j["classes"]); g2l = {g: i for i, g in enumerate(classes)}
        model = Net(ALL_MODELS[m], len(classes), pretrained=False).to(DEVICE)
        model.load_state_dict(torch.load(wf, map_location=DEVICE)); model.eval()
        for b in batches(ix, 16):
            x = eval_view(CACHE_T[b].to(DEVICE))
            with torch.no_grad():
                pred = model(x)[0].argmax(1)
            y = torch.tensor([g2l[int(g)] for g in Y_ALL[b]], device=DEVICE)
            sals = {"gradcam": gradcam(model, x, pred)}
            if ALL_MODELS[m]["head"] == "csra":
                sals["lesion_attention"] = attn_map(model, x, pred)
            sals["random"] = F.interpolate(torch.rand(len(b), 1, 7, 7, device=DEVICE), size=(IMG_SIZE, IMG_SIZE),
                                           mode="bilinear", align_corners=False).squeeze(1)
            for sname, sal in sals.items():
                ent, border = sal_stats(sal)
                for r in range(len(b)):
                    d_auc, i_auc = del_ins(model, x[r], sal[r], int(pred[r]))
                    row = dict(model=m, held_out=h, saliency=sname, i=int(b[r]), correct=bool(pred[r] == y[r]),
                               deletion_auc=d_auc, insertion_auc=i_auc, entropy=ent[r], border_mass=border[r])
                    for tag, mdir in (("A", MASKS_A), ("B", MASKS_B)):
                        mk = load_mask(mdir, int(b[r])) if mdir else None
                        if mk is not None and mk.any():
                            s_ = sal[r].cpu()
                            row[f"pointing_energy_{tag}"] = float(s_[mk].sum() / (s_.sum() + 1e-8))
                            top = s_ >= torch.quantile(s_.flatten(), 0.85)
                            row[f"iou_top15_{tag}"] = float((top & mk).sum() / ((top | mk).sum() + 1e-8))
                    XAI_ROWS.append(row)
        del model; torch.cuda.empty_cache()

### 11.7 RQ4 results

In [ ]:
xai = pd.DataFrame(XAI_ROWS)
if len(xai):
    xai.to_csv(TABLES / "xai_per_image.csv", index=False)
    agg_cols = [c for c in ["deletion_auc", "insertion_auc", "entropy", "border_mass",
                            "pointing_energy_A", "iou_top15_A"] if c in xai]
    report("RQ4 — saliency faithfulness and concentration (held-out ID images, seed 0)",
           md_table(xai.groupby(["model", "saliency"])[agg_cols].mean()))
    fa = []
    for (m, s), g in xai[xai.saliency != "random"].groupby(["model", "saliency"]):
        if g.correct.nunique() < 2:
            continue
        for metric in ["entropy", "border_mass", "deletion_auc"]:
            a, b_ = g[g.correct][metric], g[~g.correct][metric]
            fa.append(dict(model=m, saliency=s, metric=metric, correct_mean=a.mean(), wrong_mean=b_.mean(),
                           mannwhitney_p=stats.mannwhitneyu(a, b_).pvalue))
    if fa:
        fa = pd.DataFrame(fa); fa.to_csv(TABLES / "xai_failure_analysis.csv", index=False)
        report("RQ4 — do wrong cross-source predictions look elsewhere? (correct vs wrong)", md_table(fa, "{:.4f}"))
    if MASKS_A and MASKS_B:
        ious = []
        for i in [int(i) for ix in XAI_IDX.values() for i in ix]:
            a, b_ = load_mask(MASKS_A, i), load_mask(MASKS_B, i)
            if a is not None and b_ is not None and (a | b_).any():
                ious.append(float((a & b_).sum() / (a | b_).sum()))
        if ious:
            report("Inter-annotator agreement", f"mean IoU between annotators = {np.mean(ious):.3f} (n={len(ious)})")

### 11.8 Qualitative examples: baseline Grad-CAM vs SILA-Net

In [ ]:
# qualitative figure: image | baseline Grad-CAM | SILA Grad-CAM | SILA lesion attention
def overlay(ax, img, sal=None, title=""):
    ax.imshow(img)
    if sal is not None:
        ax.imshow(sal, cmap="jet", alpha=0.45)
    ax.set_title(title, fontsize=7); ax.axis("off")

shown = []
for h, ix in XAI_IDX.items():
    shown += [(h, i) for i in ix[:2]]
shown = shown[:8]
k_base = {h: run_key("EfficientNetB0", "lodo", h, SEEDS[0]) for h in XAI_IDX}
k_prop = {h: run_key(PROPOSED, "lodo", h, SEEDS[0]) for h in XAI_IDX}
if shown and all((WEIGHTS / f"{k_prop[h]}.pt").exists() and (WEIGHTS / f"{k_base[h]}.pt").exists() and run_done(k_prop[h]) and run_done(k_base[h]) for h, _ in shown):
    fig, axes = plt.subplots(len(shown), 4, figsize=(9, 2.3 * len(shown)), squeeze=False)
    for r, (h, i) in enumerate(shown):
        x = eval_view(CACHE_T[[i]].to(DEVICE)); img = x[0].permute(1, 2, 0).cpu().numpy()
        cols = [None, None, None]
        for c, (kk, mname) in enumerate([(k_base[h], "EfficientNetB0"), (k_prop[h], PROPOSED)]):
            jj = json.loads((RUNS / f"{kk}.json").read_text())
            mdl = Net(ALL_MODELS[mname], len(jj["classes"]), pretrained=False).to(DEVICE)
            mdl.load_state_dict(torch.load(WEIGHTS / f"{kk}.pt", map_location=DEVICE)); mdl.eval()
            with torch.no_grad():
                pr = mdl(x)[0].argmax(1)
            cols[c] = (gradcam(mdl, x, pr)[0].cpu().numpy(), CLASSES[jj["classes"][int(pr)]])
            if mname == PROPOSED:
                cols[2] = (attn_map(mdl, x, pr)[0].cpu().numpy(), CLASSES[jj["classes"][int(pr)]])
        overlay(axes[r, 0], img, None, f"{h}\ntrue: {idx.label_name[i]}")
        overlay(axes[r, 1], img, cols[0][0], f"EffNet-B0 Grad-CAM\npred: {cols[0][1]}")
        overlay(axes[r, 2], img, cols[1][0], f"SILA Grad-CAM\npred: {cols[1][1]}")
        overlay(axes[r, 3], img, cols[2][0], f"SILA lesion attention\npred: {cols[2][1]}")
    plt.tight_layout(); plt.savefig(FIGS / "xai_examples.png", dpi=120); plt.show()

## 12. Region × season remedy knowledge base + uncertainty-gated recommendation

**Status: template; must be verified by the team before the paper.** Each entry names the *kind* of action. Every entry has `source` and `verified` fields: fill `source` from official documents (BARI onion production guide, DAE/BAMIS crop-disease weather calendars for Bangladesh, ICAR/state agricultural university advisories for India, national/FAO extension guides for Burkina Faso), add product/dose only as the source states it, then set `verified=True`. Unverified entries are flagged in every output. KB **coverage** and the **share verified** are reported as metrics.

Seasons: Bangladesh/India: *rabi* (Oct–Mar, main onion season), *pre-kharif/zaid* (Apr–May), *kharif* (Jun–Sep). Burkina Faso: *dry season* (Nov–May, main irrigated onion season), *rainy season* (Jun–Oct).

In [ ]:
SEASONS = {
    "Bangladesh": {**{m: "rabi" for m in (10, 11, 12, 1, 2, 3)}, 4: "pre_kharif", 5: "pre_kharif",
                   **{m: "kharif" for m in (6, 7, 8, 9)}},
    "India": {**{m: "rabi" for m in (10, 11, 12, 1, 2, 3)}, 4: "zaid", 5: "zaid",
              **{m: "kharif" for m in (6, 7, 8, 9)}},
    "Burkina Faso": {**{m: "dry" for m in (11, 12, 1, 2, 3, 4, 5)}, **{m: "rainy" for m in (6, 7, 8, 9, 10)}},
}
GENERIC = {
    "none": "No disease detected. Keep routine scouting (twice weekly) and balanced fertilization.",
    "fungal_foliar": "Leaf-blight complex (purple blotch / Stemphylium / Botrytis): remove infected leaves, avoid overhead and late-day irrigation, wider spacing for airflow, rotate away from Allium; protectant + systemic foliar fungicide programme as per local guideline.",
    "oomycete": "Downy mildew: improve drainage and airflow, avoid leaf wetness, remove volunteer onions; oomycete-specific fungicide as per local guideline; avoid repeated single-mode-of-action sprays.",
    "rust": "Rust: remove infected debris and volunteer Allium, balanced nitrogen; rust-labelled fungicide as per local guideline.",
    "vector_control": "Viral (e.g. Iris yellow spot virus): no cure. Manage thrips vectors (monitoring, sticky traps, recommended insecticide rotation), rogue infected plants, control weeds/volunteer hosts.",
    "bacterial": "Bacterial leaf blight: use clean seed/sets, avoid working in wet fields, reduce leaf wetness; copper-based bactericide where the local guideline allows.",
    "soilborne": "Soil/bulb disease (Fusarium basal rot / bulb blight): long rotation, healthy treated sets/seed, good drainage, avoid bulb injury, cure and store bulbs properly.",
    "abiotic_tip": "Tip burn: check water stress, salinity, nutrient imbalance and air pollution; rule out Botrytis/early blight before spraying.",
}
SEASON_NOTE = {   # seasonal emphasis (to verify against BAMIS / ICAR / national calendars)
    ("fungal_foliar", "rabi"): "Risk rises with foggy/humid spells and cool nights in the late season; scout more often after rain or fog.",
    ("fungal_foliar", "kharif"): "Monsoon humidity: high blight pressure; protectant cover before rain events.",
    ("oomycete", "rabi"): "Cool, humid nights with long leaf wetness favour downy mildew.",
    ("vector_control", "rabi"): "Thrips populations build up in dry, warm late-season weather; intensify monitoring.",
    ("vector_control", "dry"): "Dry season favours thrips; monitor weekly.",
    ("fungal_foliar", "rainy"): "Rainy season: high leaf-wetness hours; prioritise drainage and protectant cover.",
    ("oomycete", "rainy"): "Rainy season: high downy-mildew risk.",
}

### 12.1 Build the knowledge base (template)

In [ ]:
KB = {}
for disease, grp in REMEDY_GROUP.items():
    for region, months in SEASONS.items():
        for season in sorted(set(months.values())):
            KB[(disease, region, season)] = dict(action=GENERIC[grp], season_note=SEASON_NOTE.get((grp, season), ""),
                                                 remedy_group=grp, source="TODO", verified=False)
kb_df = pd.DataFrame([dict(disease=d, region=r, season=s, **v) for (d, r, s), v in KB.items()])
kb_df.to_csv(TABLES / "remedy_knowledge_base_TEMPLATE.csv", index=False)
cov = np.mean([any(k[0] == c for k in KB) for c in CLASSES])
report("Remedy knowledge base", f"- entries: {len(KB)} (disease × region × season)\n- disease coverage of model classes: {cov:.0%}\n"
       f"- entries with a verified official source: {kb_df.verified.mean():.0%} (fill `source`, set `verified=True`)")

### 12.2 The gated recommender

In [ ]:
def recommend(probs, classes, region, month, q_remedy):
    """Remedy-level conformal gate. probs: calibrated disease probabilities over `classes` (global ids).
    A remedy is issued only if the conformal set over REMEDY GROUPS has exactly one member."""
    P = group_probs(probs[None], classes)[0]
    rset = [GROUPS[g] for g in np.where(1 - P <= q_remedy)[0]]
    top = CLASSES[classes[int(probs.argmax())]]
    region = region if region in SEASONS else "Bangladesh"
    season = SEASONS[region][month]
    if len(rset) != 1:
        return dict(status="WITHHELD", remedy_set=rset, top1=top, confidence=float(probs.max()),
                    message=("Uncertain between treatments: " + " / ".join(rset) if rset else "Image unlike anything seen in training")
                            + ". Re-take a close, well-lit photo of the affected leaf, or consult the local extension officer.")
    grp = rset[0]
    cands = [CLASSES[c] for c in classes if REMEDY_GROUP[CLASSES[c]] == grp]
    likely = max(cands, key=lambda c: probs[list(classes).index(CLASSES.index(c))])
    e = KB[(likely, region, season)]
    return dict(status="RECOMMEND", remedy_group=grp, likely_disease=likely, compatible_diseases=cands,
                remedy_confidence=float(P[G2I[grp]]), region=region, season=season,
                action=e["action"], season_note=e["season_note"], source=e["source"],
                warning="" if e["verified"] else "UNVERIFIED KB ENTRY")

### 12.3 Worked case studies

In [ ]:
# worked case studies from the proposed model (seed 0)
cases = []
for h in dss:
    k = run_key(PROPOSED, "lodo", h, SEEDS[0])
    if not run_done(k):
        continue
    qh = QHAT.get((PROPOSED, h, SEEDS[0]))
    if qh is None:
        continue
    cal = "xsrc" if "xsrc" in qh else "worst"
    q_r = qh[cal][1]
    jj, z = load_run(k); classes = np.array(z["classes"])
    pt = softmax(z["test_logits"], qh["T"])
    region = REGION[h] if REGION[h] != "unknown" else "Bangladesh"
    y_true = Y_ALL[z["test_idx"]].numpy()
    true_g = CLASS_GROUP[y_true]
    rset = (1 - group_probs(pt, classes)) <= q_r
    accept = rset.sum(1) == 1
    issued = rset.argmax(1)
    same_remedy_confusion = accept & (classes[pt.argmax(1)] != y_true) & (issued == true_g)
    wanted = {"confident & correct": accept & (classes[pt.argmax(1)] == y_true),
              "disease confused, remedy still right": same_remedy_confusion,
              "wrong remedy avoided (WITHHELD)": ~accept & (CLASS_GROUP[classes[pt.argmax(1)]] != true_g),
              "unseen disease WITHHELD": ~accept & ~np.isin(y_true, classes),
              "wrong remedy issued (failure case)": accept & (issued != true_g)}
    for kind, msk in wanted.items():
        w = np.where(msk)[0]
        if len(w):
            r = recommend(pt[w[0]], classes, region, 1, q_r)
            cases.append(dict(held_out=h, calibration=cal, kind=kind, true=CLASSES[y_true[w[0]]],
                              file=idx.rel[int(z["test_idx"][w[0]])], **r))
if cases:
    cs = pd.DataFrame(cases); cs.to_csv(TABLES / "remedy_case_studies.csv", index=False)
    report("Worked case studies - remedy-level gate (January, region of the held-out set; unknown origin -> Bangladesh)",
           md_table(cs[[c for c in ["held_out", "kind", "true", "status", "remedy_set", "likely_disease",
                                    "remedy_group", "remedy_confidence", "confidence"] if c in cs]].fillna("")))

## 13. Efficiency (on-device readiness)

Parameters, GFLOPs (224², batch 1), median CPU latency (4 threads; a rough stand-in for a mid-range phone) and GPU latency; ONNX export of SILA-Net for later TFLite conversion (Phase 4).

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
try:
    import onnx  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx"], check=False)
eff = []
for m in MAIN_MODELS:
    if ALL_MODELS[m]["head"] == "probe":     # frozen foundation encoders: parameter count only (not phone-sized)
        pm = FM_PARAMS.get(m, RI[RI.model == m].params_m.iloc[0] if len(RI) and (RI.model == m).any() else np.nan)
        eff.append(dict(model=m, params_M=pm))
        continue
    if m in DG_MODELS:                       # same architecture/cost as EfficientNetB0
        continue
    mdl = Net(ALL_MODELS[m], len(CLASSES), pretrained=False).eval()
    x = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
    with FlopCounterMode(display=False) as fc:
        mdl(x)
    torch.set_num_threads(4)
    with torch.no_grad():
        for _ in range(3): mdl(x)
        ts = []
        for _ in range(15 if not SMOKE else 3):
            t = time.perf_counter(); mdl(x); ts.append(time.perf_counter() - t)
    row = dict(model=m, params_M=count_params(mdl), GFLOPs=fc.get_total_flops() / 2e9, cpu_ms=1000 * np.median(ts))
    if AMP:
        mg, xg = mdl.to(DEVICE), x.to(DEVICE)
        with torch.no_grad():
            for _ in range(5): mg(xg)
            torch.cuda.synchronize(); t = time.perf_counter()
            for _ in range(30): mg(xg)
            torch.cuda.synchronize()
        row["gpu_ms"] = 1000 * (time.perf_counter() - t) / 30
    eff.append(row)
    if m == PROPOSED:
        try:
            class Export(nn.Module):
                def __init__(s, n): super().__init__(); s.n = n
                def forward(s, x): return s.n(x)[0].softmax(1)
            torch.onnx.export(Export(mdl.cpu()).eval(), x, str(RES / "sila_net.onnx"), input_names=["image01"],
                              output_names=["probs"], opset_version=17, dynamo=False)
            print("ONNX exported:", RES / "sila_net.onnx")
        except Exception as e:
            print("ONNX export skipped:", e)
    del mdl
eff = pd.DataFrame(eff).set_index("model")
eff.to_csv(TABLES / "efficiency.csv")
report("Efficiency (224×224, batch 1)", md_table(eff, "{:.2f}"))

## 14. Report and archive

In [ ]:
lines = ["# Onion cross-region diagnosis — results report", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M')} | {'SMOKE TEST' if SMOKE else 'full run'} | "
         f"runs available: {len(RUN_INDEX)} / planned {len(PLAN)} | classes: {', '.join(CLASSES)}", "",
         "Figures: " + ", ".join(sorted(p.name for p in FIGS.glob('*.png'))), ""]
for title, text in REPORT:
    lines += [f"## {title}", "", text, ""]
(RES / "REPORT.md").write_text("\n".join(lines), encoding="utf-8")
save_json(dict(SMOKE=SMOKE, MERGE_VIRAL=MERGE_VIRAL, IMG_SIZE=IMG_SIZE, EPOCHS=EPOCHS, BATCH=BATCH, SEEDS=SEEDS,
               CV_FOLDS=CV_FOLDS, DEDUP_HAMMING=DEDUP_HAMMING, ALPHA=ALPHA, CSRA_LAMBDA=CSRA_LAMBDA,
               FOURIER_BETA=FOURIER_BETA, LAMBDA_KL=LAMBDA_KL, LAMBDA_ATT=LAMBDA_ATT, classes=CLASSES,
               torch=torch.__version__, timm=timm.__version__), RES / "config.json")
with zipfile.ZipFile(OUT / "results_light.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RES.rglob("*"):
        if f.is_file() and f.suffix not in {".pt", ".npz", ".onnx", ".npy"}:
            zf.write(f, f.relative_to(OUT))
print("REPORT:", RES / "REPORT.md", "| archive:", OUT / "results_light.zip", f"| total time {hours():.2f} h")